In [4]:
#!/usr/bin/env python
# coding: utf-8
"""
NQ IB / ORB Price-Structure Pipeline -- 15y OHLCV edition
==========================================================

Ports the NON-volume analysis from the v5 tick pipeline (Untitled3.py) onto
the 15-year 1-min OHLCV file used by Orb_IB_positioning.py
(nq_15y_1min.csv: Date, Time, Open, High, Low, Close [, Volume]).

INCLUDED (everything that needs only price, plus VWAP levels):
  Stage 2 features : session context (Asia/London direction + 4-way pattern,
                     NY open vs London/overnight midpoints), ORB-15/ORB-30
                     structure (close half, direction, position in IB,
                     internal extreme-first, curve results), IB structure
                     (direction, close half/zone, extreme-first, quarter/
                     curve type), premarket (6-9am) price structure, VWAP
                     anchors (weekly / session / prev-week / premarket --
                     needs a Volume column; skipped cleanly if absent),
                     size (range %) z-score bands, break/outcome tracking,
                     10-signal structural score, trade type.
  Stage 3 analysis : Items 1-5 (direction concordance, break rates by close
                     half x extreme-first, ORB position conditioned on
                     extreme/break-first, London-midpoint position, Asia/
                     London pattern breakdown) + the 10:00-10:30 trading-
                     framework tables (look-ahead vs live-safe versions).
  Stage 4 (Part A) : size bands -> outcomes/breaks, size x close-half x
                     extreme-first, and size as a third factor on the ORB-30
                     live-safe signal.
  Stage 5          : VWAP anchors Parts A/B/C (only if Volume present).
  Stage 6          : premarket size, premarket direction <-> IB direction
                     concordance, premarket vs premarket-VWAP, and premarket
                     direction/size as a third factor on the ORB-30 signal.
  Ranked edge table across everything collected.

EXCLUDED (requires tick aggressor data, or was the volume-level analysis
that didn't pan out): delta anything (bias, POC, quartile structure,
divergence, pre-IB windows), buy/sell volume split, P/b/B/D profile shapes,
volume bands, order-flow score, Stage 7 volume diagnostics.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR      = Path(r"C:\Users\tammm\NQ and ES analysis")
RAW_DATA_FILE = "nq_15y_1min.csv"
RAW_DATA_PATH = DATA_DIR / RAW_DATA_FILE

START_DATE = "2013-01-01"
END_DATE   = "2025-12-31"

OUTPUT_DATASET_PATH = DATA_DIR / "nq_15y_price_structure_dataset.csv"
RANKED_EDGE_PATH    = DATA_DIR / "nq_15y_price_structure_edge_table.csv"

# If the dataset CSV already exists and this is False, Stage 2 is skipped
# and the analysis runs straight off the cached CSV (much faster on reruns).
REBUILD_DATASET = True

# Retest analysis: for ORB-15/ORB-30 days with a clean direction2 signal,
# scan the window from that ORB's close to IB completion (10:30) for the
# first touch of a handful of reference levels, then check whether IB
# direction still confirms the bias conditional on that touch happening.
RUN_RETEST_ANALYSIS = True
RETEST_OUTPUT_PATH  = DATA_DIR / "nq_15y_orb_retest_dataset.csv"
RETEST_EDGE_PATH    = DATA_DIR / "nq_15y_orb_retest_edge_table.csv"

MIN_ORB_RANGE_FRACTION   = 0.03   # ORB range >= this fraction of prev day's range
MIN_STRUCTURAL_AGREEMENT = 6      # out of 10 structural signals
MIN_DISPLAY_SAMPLE       = 30

BAND_ORDER = ['Very Small', 'Small', 'Average', 'Large', 'Very Large']
buckets = ['bottom_third', 'middle_third', 'top_third']

# ============================================================
# HELPERS -- structural (ported unchanged from the v5 pipeline)
# ============================================================

def close_half(close, high, low):
    if high == low:
        return None
    mid = (high + low) / 2
    return 'upper' if close >= mid else 'lower'


def position_in_range(price, high, low):
    if high == low:
        return None
    return (price - low) / (high - low)


def position_bucket(score):
    if score is None:
        return None
    if score <= 1 / 3:
        return 'bottom_third'
    if score <= 2 / 3:
        return 'middle_third'
    return 'top_third'


def zone_label(pct):
    if pct < 0.25:
        return 'Q1'
    if pct < 0.50:
        return 'Q2'
    if pct < 0.75:
        return 'Q3'
    return 'Q4'


def session_direction(o, c):
    if o == c:
        return None
    return 'bullish' if c > o else 'bearish'


def classify_asia_london(london_high, london_low, asia_high, asia_low):
    """Four-way exhaustive taxonomy (matches the v4/v5 scripts)."""
    if london_high > asia_high and london_low < asia_low:
        return 'london_engulfs_asia'
    elif london_high <= asia_high and london_low >= asia_low:
        return 'asia_engulfs_london'
    elif london_high > asia_high and london_low >= asia_low:
        return 'london_partially_engulfs_upwards'
    elif london_low < asia_low and london_high <= asia_high:
        return 'london_partially_engulfs_downwards'
    else:
        return 'other'


def which_quarter(ts, q1_end, q2_end, q3_end):
    if ts < q1_end:
        return 1
    if ts < q2_end:
        return 2
    if ts < q3_end:
        return 3
    return 4


def which_extreme_first(data, ref_high, ref_low):
    """Scan a slice chronologically, report whether ref_high or ref_low was
    touched first. Used for the IB's final extreme-first (look-ahead if the
    true extreme prints in Q3/Q4) and for each ORB's OWN internal
    extreme-first (fully known when that ORB's window closes)."""
    for _, row in data.iterrows():
        hit_high = row['high'] >= ref_high
        hit_low = row['low'] <= ref_low
        if hit_high and hit_low:
            return 'simultaneous'
        elif hit_high:
            return 'high'
        elif hit_low:
            return 'low'
    return None


def find_first_break(data, ref_high, ref_low):
    h_broken = l_broken = False
    h_time = l_time = None
    for idx, row in data.iterrows():
        if not h_broken and row['high'] > ref_high:
            h_broken, h_time = True, idx
        if not l_broken and row['low'] < ref_low:
            l_broken, l_time = True, idx
    return h_broken, h_time, l_broken, l_time


def compute_curve_result(breaking_high, breaking_low, ref_high, ref_low,
                          sh_high, sh_low):
    """Generic curve-vs-continuation classifier: serves ORB-15 (breaking
    block=Q2, second-half ref=Q3+Q4), ORB-30 within-IB (Q3 vs Q4) and ORB-30
    post-IB (Q3 vs full post-IB session)."""
    breaks_high = breaking_high > ref_high
    breaks_low = breaking_low < ref_low
    if breaks_high and not breaks_low:
        if sh_low < ref_low:
            return 'broke_high_curved_low'
        elif sh_high > breaking_high:
            return 'broke_high_continued_high'
        else:
            return 'broke_high_neither'
    elif breaks_low and not breaks_high:
        if sh_high > ref_high:
            return 'broke_low_curved_high'
        elif sh_low < breaking_low:
            return 'broke_low_continued_low'
        else:
            return 'broke_low_neither'
    elif breaks_high and breaks_low:
        return 'breaks_both'
    else:
        return 'contained'


def compute_trade_type(ib_extreme, ib_close_h):
    if ib_extreme is None or ib_close_h is None:
        return 'unclear'
    if ib_extreme == 'low' and ib_close_h == 'upper':
        return 'extension_long'
    if ib_extreme == 'high' and ib_close_h == 'lower':
        return 'extension_short'
    if ib_extreme == 'high' and ib_close_h == 'upper':
        return 'fade_long'
    if ib_extreme == 'low' and ib_close_h == 'lower':
        return 'fade_short'
    return 'unclear'


def extreme_close_direction(extreme_first, close_h):
    """SECOND definition of bullish/bearish (distinct from session_direction's
    open-vs-close test):
        bullish = low formed first AND close in the upper half
        bearish = high formed first AND close in the lower half
    The other two combinations (extreme-first agrees with a close on the
    SAME side -- the 'fade' patterns in compute_trade_type: high-first +
    upper-close, low-first + lower-close) don't match either clean pattern
    and are returned as None, same as 'simultaneous' or missing data.
    """
    if extreme_first == 'low' and close_h == 'upper':
        return 'bullish'
    if extreme_first == 'high' and close_h == 'lower':
        return 'bearish'
    return None


# ============================================================
# HELPERS -- signal scoring (structural only; order-flow score removed)
# ============================================================

ASIA_LONDON_BIAS = {
    'london_engulfs_asia': 'neutral',
    'asia_engulfs_london': 'neutral',
    'london_partially_engulfs_upwards': 'bullish',
    'london_partially_engulfs_downwards': 'bearish',
}


def score_dir(d):
    if d == 'bullish':
        return 'bullish'
    if d == 'bearish':
        return 'bearish'
    return 'neutral'


def score_pattern(p):
    return ASIA_LONDON_BIAS.get(p, 'neutral')


def score_close_half(ch):
    if ch == 'upper':
        return 'bullish'
    if ch == 'lower':
        return 'bearish'
    return 'neutral'


def score_position(bucket):
    if bucket == 'bottom_third':
        return 'bullish'
    if bucket == 'top_third':
        return 'bearish'
    return 'neutral'


def score_extreme_first(e):
    if e == 'low':
        return 'bullish'
    if e == 'high':
        return 'bearish'
    return 'neutral'


def score_ny_vs_mid(pos):
    if pos == 'above':
        return 'bullish'
    if pos == 'below':
        return 'bearish'
    return 'neutral'


def aggregate(scores, min_agree):
    bull = scores.count('bullish')
    bear = scores.count('bearish')
    if bull == 0 and bear == 0:
        return 'no_edge', bull, bear, False
    bias = 'bullish' if bull >= bear else 'bearish'
    return bias, bull, bear, (max(bull, bear) >= min_agree)


def vs_level(price, level):
    if pd.isna(level):
        return None
    return 'above' if price > level else 'below'


# ============================================================
# HELPERS -- printing / analysis (same conventions as Stages 3-6)
# ============================================================

def safe_pct(n, d):
    return (n / d * 100) if d > 0 else 0.0


def print_header(text, width=95):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=95):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar = "#" * int(pct / 2)
    print(f"{' '*indent}{label:<50} {count:5d}  ({pct:5.1f}%)  {bar}")


def outcome_stats(subset, direction_col='post_ib_direction', min_n=MIN_DISPLAY_SAMPLE):
    bull = (subset[direction_col] == 'bullish').sum()
    bear = (subset[direction_col] == 'bearish').sum()
    n = bull + bear
    if n < min_n:
        return None
    bull_pct = safe_pct(bull, n)
    return {'n': n, 'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
            'edge_vs_50': abs(bull_pct - 50), 'bias': 'BULL' if bull_pct > 50 else 'BEAR'}


def print_outcome(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<50} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<50} n={s['n']:4d}  "
          f"Bull:{s['bull_pct']:5.1f}%  Bear:{s['bear_pct']:5.1f}%  Edge:{s['edge_vs_50']:4.1f}%  [{s['bias']}]")


def break_stats(subset, min_n=MIN_DISPLAY_SAMPLE):
    n = len(subset)
    if n < min_n:
        return None
    hb1 = subset['ib_high_broken_1h'].sum()
    lb1 = subset['ib_low_broken_1h'].sum()
    hbs = subset['ib_high_broken_session'].sum()
    lbs = subset['ib_low_broken_session'].sum()
    return {'n': n, 'h1h': safe_pct(hb1, n), 'l1h': safe_pct(lb1, n),
            'hses': safe_pct(hbs, n), 'lses': safe_pct(lbs, n)}


def print_break(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE):
    s = break_stats(subset, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<50} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<50} n={s['n']:4d}  "
          f"H1h={s['h1h']:5.1f}%  L1h={s['l1h']:5.1f}%  Hses={s['hses']:5.1f}%  Lses={s['lses']:5.1f}%")


def direction_concordance(df, a_col, b_col, a_name, b_name, min_n=MIN_DISPLAY_SAMPLE):
    sub = df[df[a_col].notna() & df[b_col].notna()]

    bb = ((sub[a_col] == 'bullish') & (sub[b_col] == 'bullish')).sum()
    bear_b = ((sub[a_col] == 'bullish') & (sub[b_col] == 'bearish')).sum()
    b_bear = ((sub[a_col] == 'bearish') & (sub[b_col] == 'bullish')).sum()
    bearbear = ((sub[a_col] == 'bearish') & (sub[b_col] == 'bearish')).sum()
    print(f"\n  Raw joint counts (n={len(sub)}):")
    print(f"  {'':<18}{b_name+' bull':>12}{b_name+' bear':>12}{'Total':>10}")
    print(f"  {a_name+' bull':<18}{bb:>12}{bear_b:>12}{bb+bear_b:>10}")
    print(f"  {a_name+' bear':<18}{b_bear:>12}{bearbear:>12}{b_bear+bearbear:>10}")
    print(f"  {'Total':<18}{bb+b_bear:>12}{bear_b+bearbear:>12}{len(sub):>10}")

    print(f"\n  {a_name} direction -> {b_name} direction  (row %, the tradeable direction)")
    for a_val in ['bullish', 'bearish']:
        s = sub[sub[a_col] == a_val]
        n = len(s)
        if n < min_n:
            print(f"    {a_name}={a_val:<8} (n={n:4d})  below min sample")
            continue
        bull = (s[b_col] == 'bullish').sum()
        bear = (s[b_col] == 'bearish').sum()
        print(f"    {a_name}={a_val:<8} (n={n:4d})  -> {b_name} bullish {safe_pct(bull,n):5.1f}%  "
              f"{b_name} bearish {safe_pct(bear,n):5.1f}%")
    print(f"\n  {b_name} direction -> {a_name} direction  (column %, 'the vice versa' -- diagnostic, not tradeable)")
    for b_val in ['bullish', 'bearish']:
        s = sub[sub[b_col] == b_val]
        n = len(s)
        if n < min_n:
            print(f"    {b_name}={b_val:<8} (n={n:4d})  below min sample")
            continue
        bull = (s[a_col] == 'bullish').sum()
        bear = (s[a_col] == 'bearish').sum()
        print(f"    {b_name}={b_val:<8} (n={n:4d})  -> {a_name} bullish {safe_pct(bull,n):5.1f}%  "
              f"{a_name} bearish {safe_pct(bear,n):5.1f}%")


def print_position_conditional(cond_col, cond_df, pos_col, base_rates):
    for cv in ['high', 'low']:
        subset = cond_df[cond_df[cond_col] == cv]
        n = len(subset)
        print(f"\n  {cond_col}={cv} (n={n}) -> {pos_col}")
        for bucket in buckets:
            count = (subset[pos_col] == bucket).sum()
            pct = safe_pct(count, n)
            b = base_rates[bucket]
            lift = round(pct / b, 2) if b > 0 else 0
            print(f"    {bucket:<15} {count:4d}  {pct:5.1f}%  (base {b:5.1f}%, lift {lift:.2f}x)")


edge_rows = []


def collect_edge(label, subset, base_bull_pct, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        return
    edge_rows.append({'label': label, 'n': s['n'], 'bull_pct': s['bull_pct'],
                       'bear_pct': s['bear_pct'], 'edge_vs_base': abs(s['bull_pct'] - base_bull_pct),
                       'bias': s['bias']})


def zscore_band(z):
    if pd.isna(z):
        return None
    if z < -1.5:
        return 'Very Small'
    if z < -0.5:
        return 'Small'
    if z < 0.5:
        return 'Average'
    if z < 1.5:
        return 'Large'
    return 'Very Large'


# ============================================================
# STAGE 1 -- LOAD THE 15y OHLCV CSV
# ============================================================
# Same loading convention as Orb_IB_positioning.py: Date + Time columns
# combined into a naive US/Eastern wall-clock DatetimeIndex. Everything
# downstream is naive-Eastern arithmetic, exactly like the v5 pipeline
# after its tz-strip.

def load_bars():
    print_header("STAGE 1 -- LOADING 15y 1-MIN OHLCV DATA")
    print(f"\n  Loading data from: {RAW_DATA_PATH}")

    df = pd.read_csv(RAW_DATA_PATH)
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
    df.set_index('Datetime', inplace=True)
    df = df.loc[START_DATE:END_DATE]
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='first')]

    rename = {'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close'}
    # accept a few common volume column spellings; absent -> price-only mode
    for vcand in ['Volume', 'volume', 'Vol', 'vol', 'VOLUME']:
        if vcand in df.columns:
            rename[vcand] = 'volume'
            break
    df = df.rename(columns=rename)

    has_volume = 'volume' in df.columns
    keep = ['open', 'high', 'low', 'close'] + (['volume'] if has_volume else [])
    df = df[keep]

    print(f"  Loaded {len(df):,} bars | {df.index.min()} -> {df.index.max()}")
    print(f"  Volume column present: {has_volume}"
          + ("" if has_volume else "  (VWAP sections will be skipped)"))
    return df, has_volume


# ============================================================
# STAGE 2 -- BUILD THE PER-DAY PRICE-STRUCTURE DATASET
# ============================================================

def build_dataset(bars, has_volume):
    print_header("STAGE 2 -- BUILDING PER-DAY PRICE-STRUCTURE DATASET")

    # ---- VWAP preprocessing (only meaningful with a volume column) ----
    if has_volume:
        bars = bars.copy()
        bars['typical_price'] = (bars['high'] + bars['low'] + bars['close']) / 3
        bars['pv'] = bars['typical_price'] * bars['volume']

        # Weekly anchor: resets Sunday 18:00 ET, runs through the week
        dow = bars.index.dayofweek  # Monday=0 ... Sunday=6
        days_since_sunday = (dow + 1) % 7
        candidate = pd.DatetimeIndex(bars.index.normalize()
                                     - pd.to_timedelta(days_since_sunday, unit='D')
                                     + pd.Timedelta(hours=18))
        needs_adjust = candidate > bars.index
        bars['week_start'] = pd.DatetimeIndex(
            np.where(needs_adjust, candidate - pd.Timedelta(days=7), candidate))

        bars['week_cum_pv'] = bars.groupby('week_start')['pv'].cumsum()
        bars['week_cum_vol'] = bars.groupby('week_start')['volume'].cumsum()
        bars['vwap_weekly'] = bars['week_cum_pv'] / bars['week_cum_vol']

        week_summary = bars.groupby('week_start').agg(wk_pv=('pv', 'sum'),
                                                      wk_vol=('volume', 'sum'))
        week_summary['wk_vwap'] = week_summary['wk_pv'] / week_summary['wk_vol']
        week_vwap_lookup = week_summary['wk_vwap'].to_dict()

        # Session anchor: resets 9:30 ET each day
        hm = bars.index.hour * 60 + bars.index.minute
        before_930 = hm < (9 * 60 + 30)
        sess_date = np.where(before_930,
                             bars.index.normalize() - pd.Timedelta(days=1),
                             bars.index.normalize())
        bars['session_start_930'] = (pd.DatetimeIndex(sess_date)
                                     + pd.Timedelta(hours=9, minutes=30))
        bars['day_cum_pv'] = bars.groupby('session_start_930')['pv'].cumsum()
        bars['day_cum_vol'] = bars.groupby('session_start_930')['volume'].cumsum()
        bars['vwap_session'] = bars['day_cum_pv'] / bars['day_cum_vol']

        print(f"  VWAP preprocessing done: {bars['week_start'].nunique()} weeks, "
              f"{bars['session_start_930'].nunique()} sessions")
    else:
        week_vwap_lookup = {}

    def week_start_for(ts):
        dow_ = ts.dayofweek
        days_since_sunday_ = (dow_ + 1) % 7
        cand = ts.normalize() - pd.Timedelta(days=int(days_since_sunday_)) + pd.Timedelta(hours=18)
        if cand > ts:
            cand -= pd.Timedelta(days=7)
        return cand

    # ---- Trading day enumeration (date = calendar date of RTH) ----
    shifted = (bars.index - pd.Timedelta(hours=7)).normalize()
    trading_days = sorted(pd.DatetimeIndex(shifted).unique())
    print(f"  {len(trading_days)} candidate trading days")

    records = []
    skipped = 0
    prev_range = None  # carried from the previous successfully-processed day

    for i, d in enumerate(trading_days[1:], 1):  # skip first (likely partial) day
        if i % 250 == 0:
            print(f"  Day {i} / {len(trading_days) - 1} ...")

        actual = pd.Timestamp(d) + pd.Timedelta(hours=7)

        asia_start = actual.replace(hour=18) - pd.Timedelta(days=1)
        asia_end = actual.replace(hour=2)
        lon_start = actual.replace(hour=2)
        lon_end = actual.replace(hour=8)

        ib_start = actual.replace(hour=9, minute=30)
        q1_end = actual.replace(hour=9, minute=45)   # ORB-15 end
        q2_end = actual.replace(hour=10, minute=0)   # ORB-30 end
        q3_end = actual.replace(hour=10, minute=15)
        ib_end = actual.replace(hour=10, minute=30)
        ny_end = actual.replace(hour=16, minute=0)

        asia_data = bars.loc[asia_start:asia_end]
        lon_data = bars.loc[lon_start:lon_end]
        orb15_data = bars.loc[ib_start:q1_end]
        orb30_data = bars.loc[ib_start:q2_end]
        q2_data = bars.loc[q1_end:q2_end]
        q3_data = bars.loc[q2_end:q3_end]
        q4_data = bars.loc[q3_end:ib_end]
        ib_data = bars.loc[ib_start:ib_end]
        post_ib_data = bars.loc[ib_end:ny_end]
        sh_after_orb15 = bars.loc[q2_end:ib_end]        # Q3+Q4, for ORB-15 curve
        full_day_data = bars.loc[asia_start:ny_end]     # seeds next day's prev_range

        if any(x.empty for x in [asia_data, lon_data, orb15_data, orb30_data,
                                  q2_data, q3_data, q4_data, ib_data, post_ib_data]):
            skipped += 1
            continue
        if len(orb15_data) < 10:  # need most of the 15 one-minute bars present
            skipped += 1
            continue

        # ---- ORB-15 / ORB-30 raw OHLC ----
        orb15_open, orb15_close = orb15_data['open'].iloc[0], orb15_data['close'].iloc[-1]
        orb15_high, orb15_low = orb15_data['high'].max(), orb15_data['low'].min()
        orb15_range = orb15_high - orb15_low
        if pd.isna([orb15_open, orb15_close, orb15_high, orb15_low]).any():
            skipped += 1
            continue

        orb30_open, orb30_close = orb30_data['open'].iloc[0], orb30_data['close'].iloc[-1]
        orb30_high, orb30_low = orb30_data['high'].max(), orb30_data['low'].min()
        orb30_range = orb30_high - orb30_low
        if pd.isna([orb30_open, orb30_close, orb30_high, orb30_low]).any():
            skipped += 1
            continue

        orb15_range_pct = (orb15_range / orb15_open) * 100 if orb15_open else np.nan
        orb30_range_pct = (orb30_range / orb30_open) * 100 if orb30_open else np.nan

        # ---- IB structure ----
        ib_open, ib_close = ib_data['open'].iloc[0], ib_data['close'].iloc[-1]
        ib_high, ib_low = ib_data['high'].max(), ib_data['low'].min()
        ib_range = ib_high - ib_low
        if pd.isna([ib_open, ib_close, ib_high, ib_low]).any() or ib_range == 0:
            skipped += 1
            continue

        ib_mid = (ib_high + ib_low) / 2
        ib_direction = 'bullish' if ib_close > ib_open else ('bearish' if ib_close < ib_open else None)
        if ib_direction is None:
            skipped += 1
            continue
        ib_close_position = 'above_midpoint' if ib_close > ib_mid else 'below_midpoint'
        ib_close_half_val = close_half(ib_close, ib_high, ib_low)
        ib_close_pct = position_in_range(ib_close, ib_high, ib_low)
        ib_close_zone = zone_label(ib_close_pct)
        ib_range_pct = (ib_range / ib_open) * 100 if ib_open else np.nan

        # ---- VWAP anchors ----
        if has_volume:
            current_week_start = week_start_for(actual)
            prev_week_start = current_week_start - pd.Timedelta(days=7)
            vwap_prevweek = week_vwap_lookup.get(prev_week_start, np.nan)
        else:
            vwap_prevweek = np.nan

        # ---- Premarket (6-9am) -- price structure (+ premarket VWAP) ----
        premarket_data = bars.loc[actual.replace(hour=6): actual.replace(hour=9)]
        if not premarket_data.empty:
            premarket_open = premarket_data['open'].iloc[0]
            premarket_close = premarket_data['close'].iloc[-1]
            premarket_high = premarket_data['high'].max()
            premarket_low = premarket_data['low'].min()
            premarket_range = premarket_high - premarket_low
            premarket_range_pct = (premarket_range / premarket_open) * 100 if premarket_open else np.nan
            premarket_direction = session_direction(premarket_open, premarket_close)
            premarket_close_half = close_half(premarket_close, premarket_high, premarket_low)
            if has_volume and premarket_data['volume'].sum() > 0:
                vwap_premarket = premarket_data['pv'].sum() / premarket_data['volume'].sum()
            else:
                vwap_premarket = np.nan
        else:
            premarket_open = premarket_close = premarket_high = premarket_low = np.nan
            premarket_range = premarket_range_pct = np.nan
            premarket_direction = None
            premarket_close_half = None
            vwap_premarket = np.nan

        if has_volume:
            vwap_weekly_at_orb15 = orb15_data['vwap_weekly'].iloc[-1]
            vwap_weekly_at_orb30 = orb30_data['vwap_weekly'].iloc[-1]
            vwap_weekly_at_ib = ib_data['vwap_weekly'].iloc[-1]
            vwap_session_at_orb15 = orb15_data['vwap_session'].iloc[-1]
            vwap_session_at_orb30 = orb30_data['vwap_session'].iloc[-1]
            vwap_session_at_ib = ib_data['vwap_session'].iloc[-1]
        else:
            vwap_weekly_at_orb15 = vwap_weekly_at_orb30 = vwap_weekly_at_ib = np.nan
            vwap_session_at_orb15 = vwap_session_at_orb30 = vwap_session_at_ib = np.nan

        orb15_vs_vwap_weekly = vs_level(orb15_close, vwap_weekly_at_orb15)
        orb30_vs_vwap_weekly = vs_level(orb30_close, vwap_weekly_at_orb30)
        ib_vs_vwap_weekly = vs_level(ib_close, vwap_weekly_at_ib)

        orb15_vs_vwap_prevweek = vs_level(orb15_close, vwap_prevweek)
        orb30_vs_vwap_prevweek = vs_level(orb30_close, vwap_prevweek)
        ib_vs_vwap_prevweek = vs_level(ib_close, vwap_prevweek)

        orb15_vs_vwap_premarket = vs_level(orb15_close, vwap_premarket)
        orb30_vs_vwap_premarket = vs_level(orb30_close, vwap_premarket)
        ib_vs_vwap_premarket = vs_level(ib_close, vwap_premarket)

        orb15_vs_vwap_session = vs_level(orb15_close, vwap_session_at_orb15)
        orb30_vs_vwap_session = vs_level(orb30_close, vwap_session_at_orb30)
        ib_vs_vwap_session = vs_level(ib_close, vwap_session_at_ib)

        # ---- ORB meaningfulness vs. previous day's full-session range ----
        orb15_meaningful = (prev_range is not None and prev_range > 0 and
                            orb15_range >= prev_range * MIN_ORB_RANGE_FRACTION)
        orb30_meaningful = (prev_range is not None and prev_range > 0 and
                            orb30_range >= prev_range * MIN_ORB_RANGE_FRACTION)

        orb15_close_half_val = close_half(orb15_close, orb15_high, orb15_low) if orb15_meaningful else None
        orb15_direction = session_direction(orb15_open, orb15_close)
        orb15_mid = (orb15_high + orb15_low) / 2
        orb15_pos_in_ib = position_in_range(orb15_mid, ib_high, ib_low)
        orb15_pos_bucket = position_bucket(orb15_pos_in_ib)

        orb30_close_half_val = close_half(orb30_close, orb30_high, orb30_low) if orb30_meaningful else None
        orb30_direction = session_direction(orb30_open, orb30_close)
        orb30_mid = (orb30_high + orb30_low) / 2
        orb30_pos_in_ib = position_in_range(orb30_mid, ib_high, ib_low)
        orb30_pos_bucket = position_bucket(orb30_pos_in_ib)

        # ---- ORB-15 / ORB-30 internal extreme-first (live-safe) ----
        orb15_internal_extreme_first = which_extreme_first(orb15_data, orb15_high, orb15_low)
        orb30_internal_extreme_first = which_extreme_first(orb30_data, orb30_high, orb30_low)

        # ---- IB extreme first (look-ahead if extreme sets in Q3/Q4) ----
        ib_extreme_first = which_extreme_first(ib_data, ib_high, ib_low)

        # ---- SECOND bullish/bearish definition: extreme-first + close-half
        #      bullish = low first + upper close ; bearish = high first + lower close
        #      (orb15/orb30 use their own internal, live-safe extreme-first;
        #      IB uses its final extreme-first, which carries look-ahead the
        #      same way ib_extreme_first itself does) ----
        orb15_direction2 = extreme_close_direction(orb15_internal_extreme_first, orb15_close_half_val)
        orb30_direction2 = extreme_close_direction(orb30_internal_extreme_first, orb30_close_half_val)
        ib_direction2 = extreme_close_direction(ib_extreme_first, ib_close_half_val)

        # ---- IB quarter / curve type ----
        ib_high_time = ib_data[ib_data['high'] == ib_high].index[0]
        ib_low_time = ib_data[ib_data['low'] == ib_low].index[0]
        ib_high_q = which_quarter(ib_high_time, q1_end, q2_end, q3_end)
        ib_low_q = which_quarter(ib_low_time, q1_end, q2_end, q3_end)
        ib_high_half = 1 if ib_high_time < q2_end else 2
        ib_low_half = 1 if ib_low_time < q2_end else 2
        if ib_high_half != ib_low_half:
            ib_curve_type = 'high_first' if ib_high_half == 1 else 'low_first'
        elif ib_high_half == 1:
            ib_curve_type = 'both_first_half'
        else:
            ib_curve_type = 'both_second_half'

        # ---- Curve results ----
        orb15_curve = compute_curve_result(
            q2_data['high'].max(), q2_data['low'].min(),
            orb15_high, orb15_low,
            sh_after_orb15['high'].max(), sh_after_orb15['low'].min(),
        )
        orb30_curve_within_ib = compute_curve_result(
            q3_data['high'].max(), q3_data['low'].min(),
            orb30_high, orb30_low,
            q4_data['high'].max(), q4_data['low'].min(),
        )
        orb30_curve_post_ib = compute_curve_result(
            q3_data['high'].max(), q3_data['low'].min(),
            orb30_high, orb30_low,
            post_ib_data['high'].max(), post_ib_data['low'].min(),
        )

        # ---- Chain agreement flags ----
        orb15_ib_close_agree = (orb15_close_half_val is not None and
                                 orb15_close_half_val == ib_close_half_val)
        orb30_ib_close_agree = (orb30_close_half_val is not None and
                                 orb30_close_half_val == ib_close_half_val)
        extreme_close_agree = None
        if ib_extreme_first in ('high', 'low'):
            expected = 'lower' if ib_extreme_first == 'high' else 'upper'
            extreme_close_agree = (ib_close_half_val == expected)

        # ---- Session context ----
        asia_open, asia_close = asia_data['open'].iloc[0], asia_data['close'].iloc[-1]
        asia_high, asia_low = asia_data['high'].max(), asia_data['low'].min()
        asia_dir = session_direction(asia_open, asia_close)

        lon_open, lon_close = lon_data['open'].iloc[0], lon_data['close'].iloc[-1]
        lon_high, lon_low = lon_data['high'].max(), lon_data['low'].min()
        lon_dir = session_direction(lon_open, lon_close)

        if pd.isna([asia_high, asia_low, lon_high, lon_low]).any():
            skipped += 1
            continue

        asia_london_pattern = classify_asia_london(lon_high, lon_low, asia_high, asia_low)

        lon_mid = (lon_high + lon_low) / 2
        overnight_high = max(asia_high, lon_high)
        overnight_low = min(asia_low, lon_low)
        overnight_mid = (overnight_high + overnight_low) / 2

        ny_8am_data = bars.loc[lon_end: lon_end + pd.Timedelta(minutes=1)]
        ny_8am_open = ny_8am_data['open'].iloc[0] if not ny_8am_data.empty else ib_open

        ny_open_vs_london_mid = 'above' if ny_8am_open > lon_mid else 'below'
        ny_open_vs_overnight_mid = 'above' if ny_8am_open > overnight_mid else 'below'

        orb15_vs_london_mid = 'above' if orb15_close > lon_mid else 'below'
        orb30_vs_london_mid = 'above' if orb30_close > lon_mid else 'below'
        ib_vs_london_mid = 'above' if ib_close > lon_mid else 'below'

        # ---- Post-IB outcome / break tracking ----
        post_ib_close = post_ib_data['close'].iloc[-1]
        post_ib_dir = session_direction(ib_close, post_ib_close)

        one_hour_post_ib = bars.loc[ib_end:ib_end + pd.Timedelta(hours=1)]
        hb_1h, hb_t_1h, lb_1h, lb_t_1h = find_first_break(one_hour_post_ib, ib_high, ib_low)
        hb_ses, hb_t_ses, lb_ses, lb_t_ses = find_first_break(post_ib_data, ib_high, ib_low)

        first_break_dir = None
        for _, row in post_ib_data.iterrows():
            bh = row['high'] > ib_high
            bl = row['low'] < ib_low
            if bh and bl:
                first_break_dir = 'simultaneous'
                break
            elif bh:
                first_break_dir = 'high'
                break
            elif bl:
                first_break_dir = 'low'
                break

        trade_type = compute_trade_type(ib_extreme_first, ib_close_half_val)

        # ---- Structural score (10 signals -- order-flow score removed) ----
        struct_signals = [
            score_dir(asia_dir),
            score_dir(lon_dir),
            score_pattern(asia_london_pattern),
            score_close_half(orb15_close_half_val),
            score_position(orb15_pos_bucket),
            score_close_half(orb30_close_half_val),
            score_position(orb30_pos_bucket),
            score_extreme_first(ib_extreme_first),
            score_close_half(ib_close_half_val),
            score_ny_vs_mid(ny_open_vs_london_mid),
        ]
        struct_bias, struct_bull, struct_bear, struct_agreement = aggregate(
            struct_signals, MIN_STRUCTURAL_AGREEMENT)
        structural_score_label = struct_bias if struct_agreement else 'no_edge'

        record = {
            'date': actual.date(),
            'prev_range': prev_range,

            # session context
            'asia_direction': asia_dir, 'london_direction': lon_dir,
            'asia_london_pattern': asia_london_pattern,
            'ny_open_vs_london_mid': ny_open_vs_london_mid,
            'ny_open_vs_overnight_mid': ny_open_vs_overnight_mid,
            'lon_high': lon_high, 'lon_low': lon_low, 'lon_mid': lon_mid,
            'asia_high': asia_high, 'asia_low': asia_low,
            'overnight_high': overnight_high, 'overnight_low': overnight_low,
            'overnight_mid': overnight_mid,

            # VWAP anchors
            'vwap_prevweek': vwap_prevweek, 'vwap_premarket': vwap_premarket,
            'vwap_weekly_at_orb15': vwap_weekly_at_orb15, 'vwap_weekly_at_orb30': vwap_weekly_at_orb30,
            'vwap_weekly_at_ib': vwap_weekly_at_ib,
            'vwap_session_at_orb15': vwap_session_at_orb15, 'vwap_session_at_orb30': vwap_session_at_orb30,
            'vwap_session_at_ib': vwap_session_at_ib,
            'orb15_vs_vwap_weekly': orb15_vs_vwap_weekly, 'orb30_vs_vwap_weekly': orb30_vs_vwap_weekly,
            'ib_vs_vwap_weekly': ib_vs_vwap_weekly,
            'orb15_vs_vwap_prevweek': orb15_vs_vwap_prevweek, 'orb30_vs_vwap_prevweek': orb30_vs_vwap_prevweek,
            'ib_vs_vwap_prevweek': ib_vs_vwap_prevweek,
            'orb15_vs_vwap_premarket': orb15_vs_vwap_premarket, 'orb30_vs_vwap_premarket': orb30_vs_vwap_premarket,
            'ib_vs_vwap_premarket': ib_vs_vwap_premarket,
            'orb15_vs_vwap_session': orb15_vs_vwap_session, 'orb30_vs_vwap_session': orb30_vs_vwap_session,
            'ib_vs_vwap_session': ib_vs_vwap_session,

            # 6-9am premarket -- price structure only
            'premarket_open': premarket_open, 'premarket_close': premarket_close,
            'premarket_high': premarket_high, 'premarket_low': premarket_low,
            'premarket_range': premarket_range, 'premarket_range_pct': premarket_range_pct,
            'premarket_direction': premarket_direction, 'premarket_close_half': premarket_close_half,

            # ORB-15
            'orb15_open': orb15_open, 'orb15_close': orb15_close,
            'orb15_high': orb15_high, 'orb15_low': orb15_low,
            'orb15_vs_london_mid': orb15_vs_london_mid,
            'orb15_range': orb15_range, 'orb15_range_pct': orb15_range_pct,
            'orb15_meaningful': orb15_meaningful,
            'orb15_direction': orb15_direction, 'orb15_close_half': orb15_close_half_val,
            'orb15_pos_in_ib': orb15_pos_in_ib, 'orb15_pos_bucket': orb15_pos_bucket,
            'orb15_curve_result': orb15_curve,
            'orb15_ib_close_agree': orb15_ib_close_agree,
            'orb15_internal_extreme_first': orb15_internal_extreme_first,
            'orb15_direction2': orb15_direction2,

            # ORB-30
            'orb30_open': orb30_open, 'orb30_close': orb30_close,
            'orb30_high': orb30_high, 'orb30_low': orb30_low,
            'orb30_vs_london_mid': orb30_vs_london_mid,
            'orb30_range': orb30_range, 'orb30_range_pct': orb30_range_pct,
            'orb30_meaningful': orb30_meaningful,
            'orb30_direction': orb30_direction, 'orb30_close_half': orb30_close_half_val,
            'orb30_pos_in_ib': orb30_pos_in_ib, 'orb30_pos_bucket': orb30_pos_bucket,
            'orb30_curve_result_within_ib': orb30_curve_within_ib,
            'orb30_curve_result_post_ib': orb30_curve_post_ib,
            'orb30_ib_close_agree': orb30_ib_close_agree,
            'orb30_internal_extreme_first': orb30_internal_extreme_first,
            'orb30_direction2': orb30_direction2,

            # IB structure
            'ib_open': ib_open, 'ib_close': ib_close, 'ib_high': ib_high, 'ib_low': ib_low,
            'ib_range': ib_range, 'ib_range_pct': ib_range_pct, 'ib_mid': ib_mid,
            'ib_direction': ib_direction, 'ib_close_position': ib_close_position,
            'ib_close_half': ib_close_half_val, 'ib_close_pct': ib_close_pct,
            'ib_close_zone': ib_close_zone,
            'ib_vs_london_mid': ib_vs_london_mid,
            'ib_extreme_first': ib_extreme_first,
            'ib_direction2': ib_direction2,
            'ib_curve_type': ib_curve_type,
            'ib_high_quarter': ib_high_q, 'ib_low_quarter': ib_low_q,
            'extreme_close_agree': extreme_close_agree,

            # breaks / outcome
            'ib_high_broken_1h': hb_1h, 'ib_low_broken_1h': lb_1h,
            'ib_high_broken_session': hb_ses, 'ib_low_broken_session': lb_ses,
            'first_break_dir': first_break_dir,
            'post_ib_direction': post_ib_dir,

            # scoring
            'structural_bull_count': struct_bull, 'structural_bear_count': struct_bear,
            'structural_score_label': structural_score_label,
            'trade_type': trade_type,
        }
        records.append(record)

        # carry this day's full-session range forward as next day's prev_range
        prev_range = full_day_data['high'].max() - full_day_data['low'].min()

    rdf = pd.DataFrame(records)
    print(f"\nBuilt {len(rdf):,} daily records | skipped {skipped}")

    # ---- Size (range %) z-scores, percentiles, bands ----
    SIZE_COLS = ['orb15_range_pct', 'orb30_range_pct', 'ib_range_pct', 'premarket_range_pct']
    for col in SIZE_COLS:
        rdf[f'{col}_zscore'] = scipy_stats.zscore(rdf[col], nan_policy='omit')
        rdf[f'{col}_pctile'] = rdf[col].rank(pct=True) * 100
        rdf[f'{col}_band'] = rdf[f'{col}_zscore'].apply(zscore_band)

    print("\nSize band distribution (IB range %):")
    print(rdf['ib_range_pct_band'].value_counts())
    print("\nStructural score distribution:")
    print(rdf['structural_score_label'].value_counts())

    rdf.to_csv(OUTPUT_DATASET_PATH, index=False)
    print(f"\nSaved unified dataset -> {OUTPUT_DATASET_PATH}")
    return rdf


# ============================================================
# STAGE 3 -- ANALYSIS
# ============================================================

def run_analysis(rdf, has_volume):
    total_days = len(rdf)

    BASE_STATS = outcome_stats(rdf, min_n=1)
    BASE_BULL_PCT = BASE_STATS['bull_pct'] if BASE_STATS else 50.0
    print(f"\n  Baseline post-IB direction: Bull {BASE_BULL_PCT:.1f}%  (n={BASE_STATS['n'] if BASE_STATS else 0})")

    IB_BASE_STATS = outcome_stats(rdf, direction_col='ib_direction', min_n=1)
    BASE_IB_BULL_PCT = IB_BASE_STATS['bull_pct'] if IB_BASE_STATS else 50.0
    print(f"  Baseline IB direction: Bull {BASE_IB_BULL_PCT:.1f}%  (n={IB_BASE_STATS['n'] if IB_BASE_STATS else 0})")

    ef = rdf[rdf['ib_extreme_first'].isin(['high', 'low'])]
    fb = rdf[rdf['first_break_dir'].isin(['high', 'low'])]
    meaningful15 = rdf[rdf['orb15_meaningful'] == True]
    meaningful30 = rdf[rdf['orb30_meaningful'] == True]

    # ========================================================
    # ITEM 1 -- ORB DIRECTION <-> IB DIRECTION
    # ========================================================
    print_header("ITEM 1 -- IF THE ORB IS BULLISH, IS THE IB BULLISH? (AND VICE VERSA)")
    direction_concordance(rdf, 'orb15_direction', 'ib_direction', 'ORB-15', 'IB')
    direction_concordance(rdf, 'orb30_direction', 'ib_direction', 'ORB-30', 'IB')

    # ========================================================
    # ITEM 1B -- SECOND BULLISH/BEARISH DEFINITION
    # ========================================================
    # direction  (Item 1 above) = open vs close, ignores where in the range
    #            the close sits or which extreme printed first.
    # direction2 (here)         = bullish only if the low formed first AND
    #            the close sits in the upper half; bearish only if the high
    #            formed first AND the close sits in the lower half. Any other
    #            combination (extreme-first/close-half pointing the same way,
    #            simultaneous extremes, or a non-meaningful/absent ORB) is
    #            neither and comes through as None -- so n here will be
    #            noticeably smaller than Item 1's n, by construction.
    print_header("ITEM 1B -- SECOND DEFINITION: LOW/HIGH-FIRST + CLOSE-HALF")
    print("  bullish = low formed first AND close in upper half")
    print("  bearish = high formed first AND close in lower half")
    print("  (orb15/orb30 use their own internal live-safe extreme-first;")
    print("   IB uses its final extreme-first, same look-ahead caveat as ib_extreme_first)\n")

    for label, col in [('ORB-15', 'orb15_direction2'), ('ORB-30', 'orb30_direction2'),
                       ('IB', 'ib_direction2')]:
        n_valid = rdf[col].notna().sum()
        print_subheader(f"{label} direction2 -- distribution (n valid = {n_valid} / {total_days})")
        for val in ['bullish', 'bearish']:
            print_row(f"{label} direction2={val}", (rdf[col] == val).sum(), total_days)

    print_subheader("ORB-15 direction2 -> IB direction (concordance)")
    direction_concordance(rdf, 'orb15_direction2', 'ib_direction', 'ORB-15_dir2', 'IB')

    print_subheader("ORB-30 direction2 -> IB direction (concordance)")
    direction_concordance(rdf, 'orb30_direction2', 'ib_direction', 'ORB-30_dir2', 'IB')

    print_subheader("IB direction2 -> Post-IB direction / break rates")
    for val in ['bullish', 'bearish']:
        subset = rdf[rdf['ib_direction2'] == val]
        print_outcome(f"ib_direction2={val}", subset)
        print_break(f"ib_direction2={val}", subset)
        collect_edge(f"ib_direction2={val}", subset, BASE_BULL_PCT)

    print_subheader("ORB-15 / ORB-30 direction2 -> IB direction (as an outcome, live-safe)")
    for label, col in [('ORB-15', 'orb15_direction2'), ('ORB-30', 'orb30_direction2')]:
        for val in ['bullish', 'bearish']:
            subset = rdf[rdf[col] == val]
            print_outcome(f"{label}_direction2={val}", subset, direction_col='ib_direction')
            collect_edge(f"{label}_direction2={val}_vs_ib", subset, BASE_IB_BULL_PCT,
                        direction_col='ib_direction')

    print_subheader("Does direction2 agree with the open-vs-close direction (Item 1)?")
    for label, d2_col, d1_col in [('ORB-15', 'orb15_direction2', 'orb15_direction'),
                                  ('ORB-30', 'orb30_direction2', 'orb30_direction'),
                                  ('IB', 'ib_direction2', 'ib_direction')]:
        sub = rdf[rdf[d2_col].notna() & rdf[d1_col].notna()]
        n = len(sub)
        if n == 0:
            print(f"    {label}: no valid rows")
            continue
        agree = (sub[d2_col] == sub[d1_col]).sum()
        print(f"    {label}: direction2 agrees with open-vs-close direction on "
              f"{safe_pct(agree, n):5.1f}%  (n={n})")

    # ========================================================
    # ITEM 2 -- BREAK RATES BY IB CLOSE HALF AND EXTREME-FIRST
    # ========================================================
    print_header("ITEM 2 -- BREAK RATES BY IB CLOSE HALF AND EXTREME-FIRST")

    print_subheader("Break rates by IB close half (marginal)")
    for ch in ['upper', 'lower']:
        print_break(f"ib_close={ch}", rdf[rdf['ib_close_half'] == ch])

    print_subheader("Break rates by IB extreme-first (marginal)")
    for ext in ['high', 'low']:
        print_break(f"extreme_first={ext}", rdf[rdf['ib_extreme_first'] == ext])

    print_subheader("Break rates by IB close half x extreme-first (joint, 2x2)")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['ib_close_half'] == ch) & (rdf['ib_extreme_first'] == ext)]
            print_break(f"ib_close={ch}+extreme={ext}", subset)
            collect_edge(f"ib_close={ch}+extreme={ext}", subset, BASE_BULL_PCT)

    # ========================================================
    # ITEM 3 -- ORB POSITION IN IB, CONDITIONED ON EXTREME/BREAK-FIRST
    # ========================================================
    print_header("ITEM 3 -- ORB POSITION IN IB, CONDITIONED ON EXTREME-FIRST AND BREAK-FIRST")
    print("  (Note: this is the ORBs' position within the IB, not the IB's own structure.)\n")

    base_ef_b15 = {b: safe_pct((ef['orb15_pos_bucket'] == b).sum(), len(ef)) for b in buckets}
    base_ef_b30 = {b: safe_pct((ef['orb30_pos_bucket'] == b).sum(), len(ef)) for b in buckets}
    base_fb_b15 = {b: safe_pct((fb['orb15_pos_bucket'] == b).sum(), len(fb)) for b in buckets}
    base_fb_b30 = {b: safe_pct((fb['orb30_pos_bucket'] == b).sum(), len(fb)) for b in buckets}

    print_subheader(f"ORB-15 position, conditioned on which IB extreme formed FIRST (n={len(ef)})")
    print_position_conditional('ib_extreme_first', ef, 'orb15_pos_bucket', base_ef_b15)

    print_subheader(f"ORB-30 position, conditioned on which IB extreme formed FIRST (n={len(ef)})")
    print_position_conditional('ib_extreme_first', ef, 'orb30_pos_bucket', base_ef_b30)

    print_subheader(f"ORB-15 position, conditioned on which direction BROKE FIRST post-IB (n={len(fb)})")
    print_position_conditional('first_break_dir', fb, 'orb15_pos_bucket', base_fb_b15)

    print_subheader(f"ORB-30 position, conditioned on which direction BROKE FIRST post-IB (n={len(fb)})")
    print_position_conditional('first_break_dir', fb, 'orb30_pos_bucket', base_fb_b30)

    # ========================================================
    # TRADING FRAMEWORK -- 10:00-10:30 DECISION WINDOW
    # ========================================================
    print_header("TRADING FRAMEWORK -- 10:00-10:30 DECISION WINDOW")
    print("  Given ORB close-half + which extreme formed first, what % of the time")
    print("  does the IB itself end up bullish vs. bearish?\n")
    print("  ib_extreme_first uses the IB's FINAL high/low -- has look-ahead if the")
    print("  true extreme prints in Q3/Q4. orb30_internal_extreme_first is fully known")
    print("  at 10:00am (ORB-30's own high/low order) -- this is the live-safe version.\n")

    for label, mdf, ch_col in [('ORB-15', meaningful15, 'orb15_close_half'),
                               ('ORB-30', meaningful30, 'orb30_close_half')]:
        print_subheader(f"{label} close-half x IB extreme-first (LOOK-AHEAD) -> IB direction")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = mdf[(mdf[ch_col] == ch) & (mdf['ib_extreme_first'] == ext)]
                print_outcome(f"{label}={ch}+ib_extreme_first={ext}", subset, direction_col='ib_direction')
                collect_edge(f"{label}={ch}+ib_extreme_first={ext}", subset, BASE_BULL_PCT)

    print_subheader("ORB-30 close-half x ORB-30's OWN extreme-first (LIVE-SAFE) -> IB direction")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = meaningful30[(meaningful30['orb30_close_half'] == ch) &
                                  (meaningful30['orb30_internal_extreme_first'] == ext)]
            print_outcome(f"ORB-30={ch}+orb30_internal_extreme_first={ext}", subset, direction_col='ib_direction')
            collect_edge(f"ORB-30={ch}+orb30_internal_extreme_first={ext}", subset, BASE_BULL_PCT)

    print_subheader("ORB-15 close-half x ORB-15's OWN extreme-first (LIVE-SAFE at 9:45am) -> IB direction")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = meaningful15[(meaningful15['orb15_close_half'] == ch) &
                                  (meaningful15['orb15_internal_extreme_first'] == ext)]
            print_outcome(f"ORB-15={ch}+orb15_internal_extreme_first={ext}", subset, direction_col='ib_direction')
            collect_edge(f"ORB-15={ch}+orb15_internal_extreme_first={ext}", subset, BASE_BULL_PCT)

    # ========================================================
    # ITEM 4 -- ORB / IB POSITION RELATIVE TO LONDON MIDPOINT
    # ========================================================
    print_header("ITEM 4 -- POSITION OF ORBS / IB RELATIVE TO THE LONDON MIDPOINT")

    for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('ORB-30', 'orb30_vs_london_mid'),
                       ('IB', 'ib_vs_london_mid')]:
        print_subheader(f"{label} close vs. London midpoint")
        for pos in ['above', 'below']:
            print_row(f"{label} closes {pos} London mid", (rdf[col] == pos).sum(), total_days)
        print()
        for pos in ['above', 'below']:
            subset = rdf[rdf[col] == pos]
            print_outcome(f"{label} {pos} London mid -> Post-IB", subset)
            print_break(f"{label} {pos} London mid", subset)
            collect_edge(f"{label}_vs_london_mid={pos}", subset, BASE_BULL_PCT)

    # ========================================================
    # ITEM 5 -- ORB / IB POSITION BY SESSION STRUCTURAL PATTERN
    # ========================================================
    print_header("ITEM 5 -- ORB/IB POSITION BY ASIA/LONDON STRUCTURAL PATTERN")

    for pat in ['london_engulfs_asia', 'asia_engulfs_london',
                'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
        pat_sub = rdf[rdf['asia_london_pattern'] == pat]
        n_pat = len(pat_sub)
        print_subheader(f"{pat}  (n={n_pat})")
        if n_pat < MIN_DISPLAY_SAMPLE:
            print("    below min sample")
            continue
        for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('ORB-30', 'orb30_vs_london_mid'),
                           ('IB', 'ib_vs_london_mid')]:
            above = (pat_sub[col] == 'above').sum()
            print(f"    {label} closes above London mid: {safe_pct(above, n_pat):5.1f}%  ({above}/{n_pat})")
        print()
        print_outcome(f"{pat} -> Post-IB", pat_sub)
        print_break(pat, pat_sub)
        collect_edge(f"asia_london_pattern={pat}", pat_sub, BASE_BULL_PCT)

        for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('IB', 'ib_vs_london_mid')]:
            for pos in ['above', 'below']:
                sub2 = pat_sub[pat_sub[col] == pos]
                print_outcome(f"  {pat[:24]} + {label}={pos}", sub2)
                collect_edge(f"pattern={pat[:24]}+{label}_vs_lonmid={pos}", sub2, BASE_BULL_PCT)

    # ========================================================
    # SIZE (was Stage 4 Part A) -- IS THE ORB/IB RANGE LARGER/SMALLER THAN NORMAL?
    # ========================================================
    print_header("SIZE -- IS THE ORB/IB RANGE LARGER OR SMALLER THAN NORMAL?")

    for label, band_col in [('ORB-15', 'orb15_range_pct_band'), ('ORB-30', 'orb30_range_pct_band'),
                            ('IB', 'ib_range_pct_band')]:
        print_subheader(f"{label} range size-band distribution + outcomes")
        for band in BAND_ORDER:
            print_row(f"{label} = {band}", (rdf[band_col] == band).sum(), total_days)
        print()
        for band in BAND_ORDER:
            subset = rdf[rdf[band_col] == band]
            print_outcome(f"{label}={band} -> Post-IB", subset)
            print_break(f"{label}={band}", subset)
            collect_edge(f"{label}_size={band}", subset, BASE_BULL_PCT)

    print_subheader("IB size band x close-half x extreme-first (does size change the earlier pattern?)")
    for band in ['Small', 'Average', 'Large']:
        band_sub = rdf[rdf['ib_range_pct_band'] == band]
        print(f"\n  IB size = {band} (n={len(band_sub)})")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
                print_break(f"    ib={ch}+ext={ext}", subset)
                collect_edge(f"size={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)

    print_subheader("Does ORB-30 SIZE sharpen the live-safe ORB-30 signal?")
    print("  Baseline: ORB-30 close-half x its own internal extreme-first -> IB direction\n")
    for band in ['Small', 'Average', 'Large']:
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['orb30_range_pct_band'] == band) & (rdf['orb30_close_half'] == ch) &
                             (rdf['orb30_internal_extreme_first'] == ext)]
                label = f"orb30_size={band}+ch={ch}+ext={ext}"
                print_outcome(label, subset, direction_col='ib_direction')
                collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

    print_subheader("Baseline for comparison (no size factor)")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
            print_outcome(f"orb30={ch}+ext={ext}  (no size)", subset, direction_col='ib_direction')

    # ========================================================
    # VWAP ANCHORS (was Stage 5) -- only meaningful with a volume column
    # ========================================================
    ANCHORS = [
        ('weekly',    'Weekly (Sun 18:00 anchor, running)'),
        ('session',   'Session (9:30 anchor, running -- classic day-VWAP)'),
        ('prevweek',  "Previous week's VWAP (static)"),
        ('premarket', '6-9am premarket VWAP (static)'),
    ]
    PRICE_POINTS = [('ORB-15', 'orb15'), ('ORB-30', 'orb30'), ('IB', 'ib')]

    if has_volume:
        print_header("VWAP -- PART A: POSITION VS. EACH VWAP ANCHOR")

        for anchor, anchor_label in ANCHORS:
            print_subheader(anchor_label)
            for label, prefix in PRICE_POINTS:
                col = f'{prefix}_vs_vwap_{anchor}'
                above = (rdf[col] == 'above').sum()
                below = (rdf[col] == 'below').sum()
                valid = above + below
                print(f"\n  {label} vs {anchor} VWAP  (n valid = {valid})")
                for pos in ['above', 'below']:
                    subset = rdf[rdf[col] == pos]
                    print_outcome(f"    {label} {pos}", subset)
                    print_break(f"    {label} {pos}", subset)
                    collect_edge(f"{label}_vs_vwap_{anchor}={pos}", subset, BASE_BULL_PCT)

        print_header("VWAP -- PART B: VWAP POSITION BY ASIA/LONDON STRUCTURAL PATTERN")
        print("  (weekly and session anchors only)\n")
        for pat in ['london_engulfs_asia', 'asia_engulfs_london',
                    'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
            pat_sub = rdf[rdf['asia_london_pattern'] == pat]
            n_pat = len(pat_sub)
            print_subheader(f"{pat}  (n={n_pat})")
            if n_pat < MIN_DISPLAY_SAMPLE:
                print("    below min sample")
                continue
            for anchor in ['weekly', 'session']:
                col = f'ib_vs_vwap_{anchor}'
                above = (pat_sub[col] == 'above').sum()
                valid = (pat_sub[col].isin(['above', 'below'])).sum()
                print(f"    IB above {anchor} VWAP: {safe_pct(above, valid):5.1f}%  ({above}/{valid})")
            print()
            for anchor in ['weekly', 'session']:
                col = f'ib_vs_vwap_{anchor}'
                for pos in ['above', 'below']:
                    sub2 = pat_sub[pat_sub[col] == pos]
                    print_outcome(f"  {pat[:22]} + IB {anchor}={pos}", sub2)
                    collect_edge(f"pattern={pat[:22]}+ib_vwap_{anchor}={pos}", sub2, BASE_BULL_PCT)

        print_header("VWAP -- PART C: DOES VWAP SHARPEN THE ORB-30 TRADING-FRAMEWORK SIGNAL?")
        print("  Baseline: ORB-30 close-half x its own internal extreme-first")
        print("  -> directional read on IB direction, live-safe at 10:00am.\n")
        for anchor in ['weekly', 'session']:
            vwap_col = f'orb30_vs_vwap_{anchor}'
            print_subheader(f"ORB-30 close-half x internal extreme-first x vs. {anchor} VWAP -> IB direction")
            for ch in ['upper', 'lower']:
                for ext in ['high', 'low']:
                    for vwap_pos in ['above', 'below']:
                        subset = rdf[(rdf['orb30_close_half'] == ch) &
                                     (rdf['orb30_internal_extreme_first'] == ext) &
                                     (rdf[vwap_col] == vwap_pos)]
                        label = f"orb30={ch}+ext={ext}+vwap_{anchor}={vwap_pos}"
                        print_outcome(label, subset, direction_col='ib_direction')
                        collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

        print_subheader("For comparison: the two-factor baseline without VWAP")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
                print_outcome(f"orb30={ch}+ext={ext}  (no VWAP)", subset, direction_col='ib_direction')
    else:
        print_header("VWAP SECTIONS SKIPPED -- no volume column in the source CSV")

    # ========================================================
    # PREMARKET (was Stage 6, price parts only)
    # ========================================================
    print_header("PREMARKET (6-9AM) -- SIZE: LARGER OR SMALLER THAN NORMAL?")

    print_subheader("Premarket range size-band distribution + outcomes")
    for band in BAND_ORDER:
        print_row(f"premarket = {band}", (rdf['premarket_range_pct_band'] == band).sum(), total_days)
    print()
    for band in BAND_ORDER:
        subset = rdf[rdf['premarket_range_pct_band'] == band]
        print_outcome(f"premarket={band} -> Post-IB", subset)
        print_break(f"premarket={band}", subset)
        collect_edge(f"premarket_size={band}", subset, BASE_BULL_PCT)

    print_subheader("Premarket size band x IB close-half x IB extreme-first")
    for band in ['Small', 'Average', 'Large']:
        band_sub = rdf[rdf['premarket_range_pct_band'] == band]
        print(f"\n  premarket size = {band} (n={len(band_sub)})")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
                print_break(f"    ib={ch}+ext={ext}", subset)
                collect_edge(f"premarket_size={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)

    print_header("PREMARKET -- DOES PREMARKET DIRECTION CONCORD WITH IB DIRECTION? (AND VICE VERSA)")

    sub = rdf[rdf['premarket_direction'].notna() & rdf['ib_direction'].notna()]
    bb = ((sub['premarket_direction'] == 'bullish') & (sub['ib_direction'] == 'bullish')).sum()
    bear_b = ((sub['premarket_direction'] == 'bullish') & (sub['ib_direction'] == 'bearish')).sum()
    b_bear = ((sub['premarket_direction'] == 'bearish') & (sub['ib_direction'] == 'bullish')).sum()
    bearbear = ((sub['premarket_direction'] == 'bearish') & (sub['ib_direction'] == 'bearish')).sum()
    print(f"\n  Raw joint counts (n={len(sub)}):")
    print(f"  {'':<20}{'IB bull':>12}{'IB bear':>12}{'Total':>10}")
    print(f"  {'premarket bull':<20}{bb:>12}{bear_b:>12}{bb+bear_b:>10}")
    print(f"  {'premarket bear':<20}{b_bear:>12}{bearbear:>12}{b_bear+bearbear:>10}")
    print(f"  {'Total':<20}{bb+b_bear:>12}{bear_b+bearbear:>12}{len(sub):>10}")

    print(f"\n  premarket direction -> IB direction (row %, the tradeable direction, known by 9:00am)")
    for val in ['bullish', 'bearish']:
        s = sub[sub['premarket_direction'] == val]
        n = len(s)
        bull = (s['ib_direction'] == 'bullish').sum()
        bear = (s['ib_direction'] == 'bearish').sum()
        print(f"    premarket={val:<8} (n={n:4d})  -> IB bullish {safe_pct(bull,n):5.1f}%  "
              f"IB bearish {safe_pct(bear,n):5.1f}%")

    print(f"\n  IB direction -> premarket direction (column %, diagnostic, not tradeable)")
    for val in ['bullish', 'bearish']:
        s = sub[sub['ib_direction'] == val]
        n = len(s)
        bull = (s['premarket_direction'] == 'bullish').sum()
        bear = (s['premarket_direction'] == 'bearish').sum()
        print(f"    IB={val:<8} (n={n:4d})  -> premarket bullish {safe_pct(bull,n):5.1f}%  "
              f"premarket bearish {safe_pct(bear,n):5.1f}%")

    # premarket direction / close-half / size as outcomes vs post-IB
    print_subheader("Premarket direction / close-half -> Post-IB outcomes")
    for val in ['bullish', 'bearish']:
        subset = rdf[rdf['premarket_direction'] == val]
        print_outcome(f"premarket_direction={val}", subset)
        print_break(f"premarket_direction={val}", subset)
        collect_edge(f"premarket_direction={val}", subset, BASE_BULL_PCT)
    for val in ['upper', 'lower']:
        subset = rdf[rdf['premarket_close_half'] == val]
        print_outcome(f"premarket_close_half={val}", subset)
        collect_edge(f"premarket_close_half={val}", subset, BASE_BULL_PCT)

    print_header("PREMARKET -- DOES PREMARKET DIRECTION/SIZE SHARPEN THE ORB-30 SIGNAL?")
    print("  Baseline: ORB-30 close-half x its own internal extreme-first")
    print("  -> directional read on IB direction, live-safe at 10:00am.\n")

    print_subheader("+ premarket direction")
    for pm_dir in ['bullish', 'bearish']:
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['premarket_direction'] == pm_dir) & (rdf['orb30_close_half'] == ch) &
                             (rdf['orb30_internal_extreme_first'] == ext)]
                label = f"premarket_dir={pm_dir}+ch={ch}+ext={ext}"
                print_outcome(label, subset, direction_col='ib_direction')
                collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

    print_subheader("+ premarket size band")
    for band in ['Small', 'Average', 'Large']:
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['premarket_range_pct_band'] == band) & (rdf['orb30_close_half'] == ch) &
                             (rdf['orb30_internal_extreme_first'] == ext)]
                label = f"premarket_size={band}+ch={ch}+ext={ext}"
                print_outcome(label, subset, direction_col='ib_direction')
                collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

    print_subheader("Baseline for comparison (no premarket factor)")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
            print_outcome(f"orb30={ch}+ext={ext}  (no premarket)", subset, direction_col='ib_direction')

    # ========================================================
    # RANKED EDGE TABLE
    # ========================================================
    print_header("RANKED EDGE TABLE -- ALL PATTERNS, SORTED BY EDGE")
    print(f"  Baseline Bull%: {BASE_BULL_PCT:.1f}  |  min n={MIN_DISPLAY_SAMPLE}\n")

    edge_df = pd.DataFrame(edge_rows).drop_duplicates(subset=['label']).sort_values(
        'edge_vs_base', ascending=False)
    print(f"  {'PATTERN':<58}{'n':>6}{'BULL%':>8}{'BEAR%':>8}{'EDGE':>8}{'BIAS':>6}")
    for _, row in edge_df.iterrows():
        print(f"  {row['label']:<58}{row['n']:>6}{row['bull_pct']:>7.1f}%{row['bear_pct']:>7.1f}%"
              f"{row['edge_vs_base']:>7.1f}%{row['bias']:>6}")

    edge_df.to_csv(RANKED_EDGE_PATH, index=False)
    print(f"\nSaved -> {RANKED_EDGE_PATH}")

    print_header("ANALYSIS COMPLETE")


# ============================================================
# ORB RETEST ANALYSIS -- levels revisited after the ORB closes,
# before IB completes (10:30am)
# ============================================================
# Question: given a live-safe directional bias at the ORB close
# (direction2 = low-first+upper-close -> bullish, high-first+lower-close ->
# bearish), does waiting for a pullback INTO the ORB's own range before
# entering make the signal MORE reliable, less reliable, or just delay an
# entry that was going to work anyway? And what do you give up on days
# where no pullback ever comes?
#
# Levels tested (all already in the Stage 2 dataset, no raw-bar reconstruction
# needed for the levels themselves -- only the touch-scan needs raw bars):
#   mid          -- midpoint of the ORB's own high-low range
#   high         -- the ORB's high (breakout-retest level in a bullish setup,
#                   since 'high formed first' is excluded by definition in
#                   bullish direction2 -- the high is whatever printed AFTER
#                   the low, so this level sits inside/at the edge of the ORB)
#   low          -- the ORB's low (deepest, highest-R:R pullback level;
#                   IS the origin extreme in a bullish setup)
#   vwap_session -- session VWAP sampled at that ORB's close
#
# 'Touch' = the level fell within some 1-min bar's [low, high] at any point
# between the ORB's close and IB completion. MAE% = worst move against the
# bias after that touch (before IB completes), as a % of the ORB's own
# range -- a rough stop-sizing reference, not a full R-multiple backtest.

def build_retest_dataset(bars, rdf):
    print_header("BUILDING ORB RETEST DATASET")
    print("  Scanning ORB-close -> IB-completion (10:30am) windows for touches")
    print("  of each ORB's own mid / high / low / session-VWAP levels...\n")

    rdf = rdf.copy()
    rdf['orb15_mid'] = (rdf['orb15_high'] + rdf['orb15_low']) / 2
    rdf['orb30_mid'] = (rdf['orb30_high'] + rdf['orb30_low']) / 2

    orb_specs = [
        ('ORB-15', 'orb15_direction2', 9, 45,
         {'mid': 'orb15_mid', 'high': 'orb15_high', 'low': 'orb15_low',
          'vwap_session': 'vwap_session_at_orb15'},
         'orb15_high', 'orb15_low'),
        ('ORB-30', 'orb30_direction2', 10, 0,
         {'mid': 'orb30_mid', 'high': 'orb30_high', 'low': 'orb30_low',
          'vwap_session': 'vwap_session_at_orb30'},
         'orb30_high', 'orb30_low'),
    ]

    records = []
    for i, (_, row) in enumerate(rdf.iterrows(), 1):
        if i % 500 == 0:
            print(f"  Day {i} / {len(rdf)} ...")

        actual = pd.Timestamp(row['date']) + pd.Timedelta(hours=7)
        ib_end = actual.replace(hour=10, minute=30)

        for orb_label, bias_col, end_h, end_m, level_cols, high_col, low_col in orb_specs:
            bias = row[bias_col]
            if bias not in ('bullish', 'bearish'):
                continue  # only score days with a clean direction2 signal

            orb_range = row[high_col] - row[low_col]
            if pd.isna(orb_range) or orb_range <= 0:
                continue

            orb_end = actual.replace(hour=end_h, minute=end_m)
            window = bars.loc[orb_end + pd.Timedelta(minutes=1): ib_end]

            rec = {'date': row['date'], 'orb_label': orb_label, 'bias': bias,
                   'ib_direction': row['ib_direction'],
                   'post_ib_direction': row['post_ib_direction']}

            if window.empty:
                for name in level_cols:
                    rec[f'{name}_touched'] = False
                    rec[f'{name}_minutes'] = np.nan
                    rec[f'{name}_mae_pct'] = np.nan
                records.append(rec)
                continue

            lows = window['low'].to_numpy()
            highs = window['high'].to_numpy()
            idx = window.index
            t0 = idx[0]

            for name, col in level_cols.items():
                level = row[col]
                if pd.isna(level):
                    rec[f'{name}_touched'] = False
                    rec[f'{name}_minutes'] = np.nan
                    rec[f'{name}_mae_pct'] = np.nan
                    continue
                mask = (lows <= level) & (highs >= level)
                if not mask.any():
                    rec[f'{name}_touched'] = False
                    rec[f'{name}_minutes'] = np.nan
                    rec[f'{name}_mae_pct'] = np.nan
                    continue
                first_pos = int(np.argmax(mask))
                rec[f'{name}_touched'] = True
                rec[f'{name}_minutes'] = (idx[first_pos] - t0).total_seconds() / 60.0

                if bias == 'bullish':
                    worst = lows[first_pos:].min()
                    mae_points = max(level - worst, 0.0)
                else:
                    worst = highs[first_pos:].max()
                    mae_points = max(worst - level, 0.0)
                rec[f'{name}_mae_pct'] = (mae_points / orb_range) * 100

            records.append(rec)

    retest_df = pd.DataFrame(records)
    print(f"\n  Built {len(retest_df):,} orb-day retest records: "
          f"{retest_df['orb_label'].value_counts().to_dict()}")
    retest_df.to_csv(RETEST_OUTPUT_PATH, index=False)
    print(f"  Saved -> {RETEST_OUTPUT_PATH}")
    return retest_df


def analyze_retests(retest_df):
    print_header("ORB-15 / ORB-30 RETEST ANALYSIS")
    print("  bias = direction2 (low-first+upper-close=bullish, high-first+lower-close=bearish)")
    print("  touch = level fell within a 1-min bar's range between ORB close and IB completion")
    print("  IB_match  = % of touched days where IB direction still confirms the bias")
    print("  edge      = IB_match(touched) minus the unconditional IB_match for that bias")
    print("              (positive = waiting for this retest IMPROVES the odds;")
    print("               negative = waiting for it selects AGAINST the better trades)")
    print("  median_MAE = median worst move against you after the touch, as a % of")
    print("              that ORB's own range (rough stop-sizing reference)\n")

    LEVELS = ['mid', 'high', 'low', 'vwap_session']
    summary_rows = []

    for orb_label in ['ORB-15', 'ORB-30']:
        odf = retest_df[retest_df['orb_label'] == orb_label]
        print_header(f"{orb_label} RETEST STATISTICS", width=95)

        for bias in ['bullish', 'bearish']:
            bdf = odf[odf['bias'] == bias]
            n_total = len(bdf)
            base_match = safe_pct((bdf['ib_direction'] == bias).sum(), n_total)
            print_subheader(f"{orb_label} bias={bias}  "
                            f"(n={n_total}, unconditional IB_match={base_match:.1f}%)")

            for level in LEVELS:
                touched = bdf[bdf[f'{level}_touched'] == True]
                untouched = bdf[bdf[f'{level}_touched'] == False]
                n_t, n_u = len(touched), len(untouched)

                if n_t < MIN_DISPLAY_SAMPLE:
                    print(f"    {level:<14} n_touched={n_t:4d}  (below min sample)")
                    continue

                touch_rate = safe_pct(n_t, n_total)
                ib_match_t = safe_pct((touched['ib_direction'] == bias).sum(), n_t)
                post_match_t = safe_pct((touched['post_ib_direction'] == bias).sum(), n_t)
                edge = ib_match_t - base_match
                med_min = touched[f'{level}_minutes'].median()
                med_mae = touched[f'{level}_mae_pct'].median()

                print(f"    {level:<14} touch_rate={touch_rate:5.1f}%  n={n_t:4d}  "
                      f"IB_match={ib_match_t:5.1f}%  edge={edge:+5.1f}%  "
                      f"post_IB_match={post_match_t:5.1f}%  "
                      f"median_wait={med_min:4.1f}m  median_MAE={med_mae:5.1f}%")

                if n_u >= MIN_DISPLAY_SAMPLE:
                    ib_match_u = safe_pct((untouched['ib_direction'] == bias).sum(), n_u)
                    print(f"    {'':<14} not touched: n={n_u:4d}  IB_match={ib_match_u:5.1f}%  "
                          f"(what you'd give up waiting for this level)")

                summary_rows.append({
                    'orb': orb_label, 'bias': bias, 'level': level,
                    'n_touched': n_t, 'touch_rate': touch_rate,
                    'ib_match': ib_match_t, 'edge_vs_unconditional': edge,
                    'post_ib_match': post_match_t,
                    'median_wait_min': med_min, 'median_mae_pct': med_mae,
                })
        print()

    print_header("RANKED SUMMARY -- ALL ORB x BIAS x LEVEL COMBINATIONS, BY EDGE")
    if not summary_rows:
        print("  No level/bias combination reached MIN_DISPLAY_SAMPLE -- nothing to rank.")
        print("  (This is expected on a small test dataset; on the full 15y file each")
        print("  combination should have hundreds to low-thousands of touched days.)")
        print_header("RETEST ANALYSIS COMPLETE")
        return

    summary_df = pd.DataFrame(summary_rows).sort_values('edge_vs_unconditional', ascending=False)
    print(f"  {'ORB':<8}{'BIAS':<10}{'LEVEL':<14}{'n':>6}{'TOUCH%':>8}{'IB_MATCH%':>11}"
          f"{'EDGE':>8}{'WAIT(m)':>9}{'MAE%':>7}")
    for _, r in summary_df.iterrows():
        print(f"  {r['orb']:<8}{r['bias']:<10}{r['level']:<14}{r['n_touched']:>6}"
              f"{r['touch_rate']:>7.1f}%{r['ib_match']:>10.1f}%{r['edge_vs_unconditional']:>+7.1f}%"
              f"{r['median_wait_min']:>8.1f}{r['median_mae_pct']:>6.1f}%")

    summary_df.to_csv(RETEST_EDGE_PATH, index=False)
    print(f"\nSaved -> {RETEST_EDGE_PATH}")
    print_header("RETEST ANALYSIS COMPLETE")


# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    need_bars = REBUILD_DATASET or RUN_RETEST_ANALYSIS or not os.path.exists(OUTPUT_DATASET_PATH)
    bars = has_volume = None
    if need_bars:
        bars, has_volume = load_bars()

    if REBUILD_DATASET or not os.path.exists(OUTPUT_DATASET_PATH):
        rdf = build_dataset(bars, has_volume)
    else:
        print(f"Loading cached dataset from {OUTPUT_DATASET_PATH} ...")
        rdf = pd.read_csv(OUTPUT_DATASET_PATH, parse_dates=['date'])
        # infer whether VWAP columns were populated when the cache was built
        has_volume = rdf['ib_vs_vwap_weekly'].notna().any() \
            if 'ib_vs_vwap_weekly' in rdf.columns else False
        print(f"  {len(rdf):,} days loaded")

    run_analysis(rdf, has_volume)

    if RUN_RETEST_ANALYSIS:
        retest_df = build_retest_dataset(bars, rdf)
        analyze_retests(retest_df)


                            STAGE 1 -- LOADING 15y 1-MIN OHLCV DATA                            

  Loading data from: C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
  Loaded 4,161,727 bars | 2013-01-02 06:00:00 -> 2025-03-14 16:59:00
  Volume column present: True

                      STAGE 2 -- BUILDING PER-DAY PRICE-STRUCTURE DATASET                      
  VWAP preprocessing done: 637 weeks, 3783 sessions
  3786 candidate trading days
  Day 250 / 3785 ...
  Day 500 / 3785 ...
  Day 750 / 3785 ...
  Day 1000 / 3785 ...
  Day 1250 / 3785 ...
  Day 1500 / 3785 ...
  Day 1750 / 3785 ...
  Day 2000 / 3785 ...
  Day 2250 / 3785 ...
  Day 2500 / 3785 ...
  Day 2750 / 3785 ...
  Day 3000 / 3785 ...
  Day 3250 / 3785 ...
  Day 3500 / 3785 ...
  Day 3750 / 3785 ...

Built 3,109 daily records | skipped 676

Size band distribution (IB range %):
Average       1353
Small         1074
Large          448
Very Large     232
Very Small       2
Name: ib_range_pct_band, dtype: int64

Structural sc

    ib_close=upper                                     n=1700  H1h= 64.8%  L1h= 13.4%  Hses= 82.5%  Lses= 36.7%
    ib_close=lower                                     n=1409  H1h= 13.0%  L1h= 60.8%  Hses= 38.9%  Lses= 76.2%

-----------------------------------------------------------------------------------------------
  Break rates by IB extreme-first (marginal)
-----------------------------------------------------------------------------------------------
    extreme_first=high                                 n=1546  H1h= 20.8%  L1h= 52.5%  Hses= 45.6%  Lses= 69.7%
    extreme_first=low                                  n=1562  H1h= 61.6%  L1h= 17.4%  Hses= 79.6%  Lses= 39.7%

-----------------------------------------------------------------------------------------------
  Break rates by IB close half x extreme-first (joint, 2x2)
-----------------------------------------------------------------------------------------------
    ib_close=upper+extreme=high                        n= 374

      asia_engulfs_london + ORB-15=above               n= 209  Bull: 54.1%  Bear: 45.9%  Edge: 4.1%  [BULL]
      asia_engulfs_london + ORB-15=below               n= 148  Bull: 57.4%  Bear: 42.6%  Edge: 7.4%  [BULL]
      asia_engulfs_london + IB=above                   n= 199  Bull: 56.3%  Bear: 43.7%  Edge: 6.3%  [BULL]
      asia_engulfs_london + IB=below                   n= 158  Bull: 54.4%  Bear: 45.6%  Edge: 4.4%  [BULL]

-----------------------------------------------------------------------------------------------
  london_partially_engulfs_upwards  (n=1311)
-----------------------------------------------------------------------------------------------
    ORB-15 closes above London mid:  59.5%  (780/1311)
    ORB-30 closes above London mid:  59.7%  (783/1311)
    IB closes above London mid:  56.7%  (743/1311)

    london_partially_engulfs_upwards -> Post-IB        n=1308  Bull: 56.6%  Bear: 43.4%  Edge: 6.6%  [BULL]
    london_partially_engulfs_upwards                   n=131

    orb30_size=Small+ch=upper+ext=low                  n= 461  Bull: 81.1%  Bear: 18.9%  Edge:31.1%  [BULL]
    orb30_size=Small+ch=lower+ext=high                 n= 334  Bull: 25.1%  Bear: 74.9%  Edge:24.9%  [BEAR]
    orb30_size=Small+ch=lower+ext=low                  n= 108  Bull: 43.5%  Bear: 56.5%  Edge: 6.5%  [BEAR]
    orb30_size=Average+ch=upper+ext=high               n= 137  Bull: 46.0%  Bear: 54.0%  Edge: 4.0%  [BEAR]
    orb30_size=Average+ch=upper+ext=low                n= 652  Bull: 83.7%  Bear: 16.3%  Edge:33.7%  [BULL]
    orb30_size=Average+ch=lower+ext=high               n= 537  Bull: 14.9%  Bear: 85.1%  Edge:35.1%  [BEAR]
    orb30_size=Average+ch=lower+ext=low                n=  62  Bull: 56.5%  Bear: 43.5%  Edge: 6.5%  [BULL]
    orb30_size=Large+ch=upper+ext=high                 n=  31  Bull: 67.7%  Bear: 32.3%  Edge:17.7%  [BULL]
    orb30_size=Large+ch=upper+ext=low                  n= 177  Bull: 87.6%  Bear: 12.4%  Edge:37.6%  [BULL]
    orb30_size=Large+ch=lowe

    orb30=lower+ext=high+vwap_session=above            n=  49  Bull: 24.5%  Bear: 75.5%  Edge:25.5%  [BEAR]
    orb30=lower+ext=high+vwap_session=below            n=1131  Bull: 16.6%  Bear: 83.4%  Edge:33.4%  [BEAR]
    orb30=lower+ext=low+vwap_session=above             n=   9  (below min sample)
    orb30=lower+ext=low+vwap_session=below             n= 197  Bull: 48.2%  Bear: 51.8%  Edge: 1.8%  [BEAR]

-----------------------------------------------------------------------------------------------
  For comparison: the two-factor baseline without VWAP
-----------------------------------------------------------------------------------------------
    orb30=upper+ext=high  (no VWAP)                    n= 340  Bull: 47.9%  Bear: 52.1%  Edge: 2.1%  [BEAR]
    orb30=upper+ext=low  (no VWAP)                     n=1379  Bull: 83.8%  Bear: 16.2%  Edge:33.8%  [BULL]
    orb30=lower+ext=high  (no VWAP)                    n=1180  Bull: 16.9%  Bear: 83.1%  Edge:33.1%  [BEAR]
    orb30=lower+ext=lo

  Day 500 / 3109 ...
  Day 1000 / 3109 ...
  Day 1500 / 3109 ...
  Day 2000 / 3109 ...
  Day 2500 / 3109 ...
  Day 3000 / 3109 ...

  Built 5,085 orb-day retest records: {'ORB-30': 2559, 'ORB-15': 2526}
  Saved -> C:\Users\tammm\NQ and ES analysis\nq_15y_orb_retest_dataset.csv

                                ORB-15 / ORB-30 RETEST ANALYSIS                                
  bias = direction2 (low-first+upper-close=bullish, high-first+lower-close=bearish)
  touch = level fell within a 1-min bar's range between ORB close and IB completion
  IB_match  = % of touched days where IB direction still confirms the bias
  edge      = IB_match(touched) minus the unconditional IB_match for that bias
              (positive = waiting for this retest IMPROVES the odds;
               negative = waiting for it selects AGAINST the better trades)
  median_MAE = median worst move against you after the touch, as a % of
              that ORB's own range (rough stop-sizing reference)


                   

In [5]:
#!/usr/bin/env python
# coding: utf-8
"""
NQ IB / ORB Price-Structure Pipeline -- 15y OHLCV edition
==========================================================

Ports the NON-volume analysis from the v5 tick pipeline (Untitled3.py) onto
the 15-year 1-min OHLCV file used by Orb_IB_positioning.py
(nq_15y_1min.csv: Date, Time, Open, High, Low, Close [, Volume]).

INCLUDED (everything that needs only price, plus VWAP levels):
  Stage 2 features : session context (Asia/London direction + 4-way pattern,
                     NY open vs London/overnight midpoints), ORB-15/ORB-30
                     structure (close half, direction, position in IB,
                     internal extreme-first, curve results), IB structure
                     (direction, close half/zone, extreme-first, quarter/
                     curve type), premarket (6-9am) price structure, VWAP
                     anchors (weekly / session / prev-week / premarket --
                     needs a Volume column; skipped cleanly if absent),
                     size (range %) z-score bands, break/outcome tracking,
                     10-signal structural score, trade type.
  Stage 3 analysis : Items 1-5 (direction concordance, break rates by close
                     half x extreme-first, ORB position conditioned on
                     extreme/break-first, London-midpoint position, Asia/
                     London pattern breakdown) + the 10:00-10:30 trading-
                     framework tables (look-ahead vs live-safe versions).
  Stage 4 (Part A) : size bands -> outcomes/breaks, size x close-half x
                     extreme-first, and size as a third factor on the ORB-30
                     live-safe signal.
  Stage 5          : VWAP anchors Parts A/B/C (only if Volume present).
  Stage 6          : premarket size, premarket direction <-> IB direction
                     concordance, premarket vs premarket-VWAP, and premarket
                     direction/size as a third factor on the ORB-30 signal.
  Ranked edge table across everything collected.

EXCLUDED (requires tick aggressor data, or was the volume-level analysis
that didn't pan out): delta anything (bias, POC, quartile structure,
divergence, pre-IB windows), buy/sell volume split, P/b/B/D profile shapes,
volume bands, order-flow score, Stage 7 volume diagnostics.
"""

import os
from collections import defaultdict
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR      = Path(r"C:\Users\tammm\NQ and ES analysis")
RAW_DATA_FILE = "nq_15y_1min.csv"
RAW_DATA_PATH = DATA_DIR / RAW_DATA_FILE

START_DATE = "2013-01-01"
END_DATE   = "2025-12-31"

OUTPUT_DATASET_PATH = DATA_DIR / "nq_15y_price_structure_dataset.csv"
RANKED_EDGE_PATH    = DATA_DIR / "nq_15y_price_structure_edge_table.csv"

# If the dataset CSV already exists and this is False, Stage 2 is skipped
# and the analysis runs straight off the cached CSV (much faster on reruns).
REBUILD_DATASET = True

# Retest analysis: for ORB-15/ORB-30 days with a clean direction2 signal,
# scan the window from that ORB's close to IB completion (10:30) for the
# first touch of a handful of reference levels, then check whether IB
# direction still confirms the bias conditional on that touch happening.
RUN_RETEST_ANALYSIS = True
RETEST_OUTPUT_PATH  = DATA_DIR / "nq_15y_orb_retest_dataset.csv"
RETEST_EDGE_PATH    = DATA_DIR / "nq_15y_orb_retest_edge_table.csv"

# ORB-30 breakout-retest backtest: entry = first retest of ORB-30's own far
# extreme (high for bullish, low for bearish) between ORB-30 close and IB
# completion. Tests a grid of stop distances (as a fraction of ORB-30 range)
# x target R-multiples x two exit windows (IB completion / full session).
RUN_ORB30_BACKTEST   = True
BACKTEST_RESULTS_PATH = DATA_DIR / "nq_15y_orb30_backtest_results.csv"
BACKTEST_STOP_FRACTIONS   = [0.15, 0.25, 0.35, 0.50]
BACKTEST_TARGET_R_MULTIPLES = [1.0, 1.5, 2.0, 3.0]

MIN_ORB_RANGE_FRACTION   = 0.03   # ORB range >= this fraction of prev day's range
MIN_STRUCTURAL_AGREEMENT = 6      # out of 10 structural signals
MIN_DISPLAY_SAMPLE       = 30

BAND_ORDER = ['Very Small', 'Small', 'Average', 'Large', 'Very Large']
buckets = ['bottom_third', 'middle_third', 'top_third']

# ============================================================
# HELPERS -- structural (ported unchanged from the v5 pipeline)
# ============================================================

def close_half(close, high, low):
    if high == low:
        return None
    mid = (high + low) / 2
    return 'upper' if close >= mid else 'lower'


def position_in_range(price, high, low):
    if high == low:
        return None
    return (price - low) / (high - low)


def position_bucket(score):
    if score is None:
        return None
    if score <= 1 / 3:
        return 'bottom_third'
    if score <= 2 / 3:
        return 'middle_third'
    return 'top_third'


def zone_label(pct):
    if pct < 0.25:
        return 'Q1'
    if pct < 0.50:
        return 'Q2'
    if pct < 0.75:
        return 'Q3'
    return 'Q4'


def session_direction(o, c):
    if o == c:
        return None
    return 'bullish' if c > o else 'bearish'


def classify_asia_london(london_high, london_low, asia_high, asia_low):
    """Four-way exhaustive taxonomy (matches the v4/v5 scripts)."""
    if london_high > asia_high and london_low < asia_low:
        return 'london_engulfs_asia'
    elif london_high <= asia_high and london_low >= asia_low:
        return 'asia_engulfs_london'
    elif london_high > asia_high and london_low >= asia_low:
        return 'london_partially_engulfs_upwards'
    elif london_low < asia_low and london_high <= asia_high:
        return 'london_partially_engulfs_downwards'
    else:
        return 'other'


def which_quarter(ts, q1_end, q2_end, q3_end):
    if ts < q1_end:
        return 1
    if ts < q2_end:
        return 2
    if ts < q3_end:
        return 3
    return 4


def which_extreme_first(data, ref_high, ref_low):
    """Scan a slice chronologically, report whether ref_high or ref_low was
    touched first. Used for the IB's final extreme-first (look-ahead if the
    true extreme prints in Q3/Q4) and for each ORB's OWN internal
    extreme-first (fully known when that ORB's window closes)."""
    for _, row in data.iterrows():
        hit_high = row['high'] >= ref_high
        hit_low = row['low'] <= ref_low
        if hit_high and hit_low:
            return 'simultaneous'
        elif hit_high:
            return 'high'
        elif hit_low:
            return 'low'
    return None


def find_first_break(data, ref_high, ref_low):
    h_broken = l_broken = False
    h_time = l_time = None
    for idx, row in data.iterrows():
        if not h_broken and row['high'] > ref_high:
            h_broken, h_time = True, idx
        if not l_broken and row['low'] < ref_low:
            l_broken, l_time = True, idx
    return h_broken, h_time, l_broken, l_time


def compute_curve_result(breaking_high, breaking_low, ref_high, ref_low,
                          sh_high, sh_low):
    """Generic curve-vs-continuation classifier: serves ORB-15 (breaking
    block=Q2, second-half ref=Q3+Q4), ORB-30 within-IB (Q3 vs Q4) and ORB-30
    post-IB (Q3 vs full post-IB session)."""
    breaks_high = breaking_high > ref_high
    breaks_low = breaking_low < ref_low
    if breaks_high and not breaks_low:
        if sh_low < ref_low:
            return 'broke_high_curved_low'
        elif sh_high > breaking_high:
            return 'broke_high_continued_high'
        else:
            return 'broke_high_neither'
    elif breaks_low and not breaks_high:
        if sh_high > ref_high:
            return 'broke_low_curved_high'
        elif sh_low < breaking_low:
            return 'broke_low_continued_low'
        else:
            return 'broke_low_neither'
    elif breaks_high and breaks_low:
        return 'breaks_both'
    else:
        return 'contained'


def compute_trade_type(ib_extreme, ib_close_h):
    if ib_extreme is None or ib_close_h is None:
        return 'unclear'
    if ib_extreme == 'low' and ib_close_h == 'upper':
        return 'extension_long'
    if ib_extreme == 'high' and ib_close_h == 'lower':
        return 'extension_short'
    if ib_extreme == 'high' and ib_close_h == 'upper':
        return 'fade_long'
    if ib_extreme == 'low' and ib_close_h == 'lower':
        return 'fade_short'
    return 'unclear'


def extreme_close_direction(extreme_first, close_h):
    """SECOND definition of bullish/bearish (distinct from session_direction's
    open-vs-close test):
        bullish = low formed first AND close in the upper half
        bearish = high formed first AND close in the lower half
    The other two combinations (extreme-first agrees with a close on the
    SAME side -- the 'fade' patterns in compute_trade_type: high-first +
    upper-close, low-first + lower-close) don't match either clean pattern
    and are returned as None, same as 'simultaneous' or missing data.
    """
    if extreme_first == 'low' and close_h == 'upper':
        return 'bullish'
    if extreme_first == 'high' and close_h == 'lower':
        return 'bearish'
    return None


# ============================================================
# HELPERS -- signal scoring (structural only; order-flow score removed)
# ============================================================

ASIA_LONDON_BIAS = {
    'london_engulfs_asia': 'neutral',
    'asia_engulfs_london': 'neutral',
    'london_partially_engulfs_upwards': 'bullish',
    'london_partially_engulfs_downwards': 'bearish',
}


def score_dir(d):
    if d == 'bullish':
        return 'bullish'
    if d == 'bearish':
        return 'bearish'
    return 'neutral'


def score_pattern(p):
    return ASIA_LONDON_BIAS.get(p, 'neutral')


def score_close_half(ch):
    if ch == 'upper':
        return 'bullish'
    if ch == 'lower':
        return 'bearish'
    return 'neutral'


def score_position(bucket):
    if bucket == 'bottom_third':
        return 'bullish'
    if bucket == 'top_third':
        return 'bearish'
    return 'neutral'


def score_extreme_first(e):
    if e == 'low':
        return 'bullish'
    if e == 'high':
        return 'bearish'
    return 'neutral'


def score_ny_vs_mid(pos):
    if pos == 'above':
        return 'bullish'
    if pos == 'below':
        return 'bearish'
    return 'neutral'


def aggregate(scores, min_agree):
    bull = scores.count('bullish')
    bear = scores.count('bearish')
    if bull == 0 and bear == 0:
        return 'no_edge', bull, bear, False
    bias = 'bullish' if bull >= bear else 'bearish'
    return bias, bull, bear, (max(bull, bear) >= min_agree)


def vs_level(price, level):
    if pd.isna(level):
        return None
    return 'above' if price > level else 'below'


# ============================================================
# HELPERS -- printing / analysis (same conventions as Stages 3-6)
# ============================================================

def safe_pct(n, d):
    return (n / d * 100) if d > 0 else 0.0


def print_header(text, width=95):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=95):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar = "#" * int(pct / 2)
    print(f"{' '*indent}{label:<50} {count:5d}  ({pct:5.1f}%)  {bar}")


def outcome_stats(subset, direction_col='post_ib_direction', min_n=MIN_DISPLAY_SAMPLE):
    bull = (subset[direction_col] == 'bullish').sum()
    bear = (subset[direction_col] == 'bearish').sum()
    n = bull + bear
    if n < min_n:
        return None
    bull_pct = safe_pct(bull, n)
    return {'n': n, 'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
            'edge_vs_50': abs(bull_pct - 50), 'bias': 'BULL' if bull_pct > 50 else 'BEAR'}


def print_outcome(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<50} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<50} n={s['n']:4d}  "
          f"Bull:{s['bull_pct']:5.1f}%  Bear:{s['bear_pct']:5.1f}%  Edge:{s['edge_vs_50']:4.1f}%  [{s['bias']}]")


def break_stats(subset, min_n=MIN_DISPLAY_SAMPLE):
    n = len(subset)
    if n < min_n:
        return None
    hb1 = subset['ib_high_broken_1h'].sum()
    lb1 = subset['ib_low_broken_1h'].sum()
    hbs = subset['ib_high_broken_session'].sum()
    lbs = subset['ib_low_broken_session'].sum()
    return {'n': n, 'h1h': safe_pct(hb1, n), 'l1h': safe_pct(lb1, n),
            'hses': safe_pct(hbs, n), 'lses': safe_pct(lbs, n)}


def print_break(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE):
    s = break_stats(subset, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<50} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<50} n={s['n']:4d}  "
          f"H1h={s['h1h']:5.1f}%  L1h={s['l1h']:5.1f}%  Hses={s['hses']:5.1f}%  Lses={s['lses']:5.1f}%")


def direction_concordance(df, a_col, b_col, a_name, b_name, min_n=MIN_DISPLAY_SAMPLE):
    sub = df[df[a_col].notna() & df[b_col].notna()]

    bb = ((sub[a_col] == 'bullish') & (sub[b_col] == 'bullish')).sum()
    bear_b = ((sub[a_col] == 'bullish') & (sub[b_col] == 'bearish')).sum()
    b_bear = ((sub[a_col] == 'bearish') & (sub[b_col] == 'bullish')).sum()
    bearbear = ((sub[a_col] == 'bearish') & (sub[b_col] == 'bearish')).sum()
    print(f"\n  Raw joint counts (n={len(sub)}):")
    print(f"  {'':<18}{b_name+' bull':>12}{b_name+' bear':>12}{'Total':>10}")
    print(f"  {a_name+' bull':<18}{bb:>12}{bear_b:>12}{bb+bear_b:>10}")
    print(f"  {a_name+' bear':<18}{b_bear:>12}{bearbear:>12}{b_bear+bearbear:>10}")
    print(f"  {'Total':<18}{bb+b_bear:>12}{bear_b+bearbear:>12}{len(sub):>10}")

    print(f"\n  {a_name} direction -> {b_name} direction  (row %, the tradeable direction)")
    for a_val in ['bullish', 'bearish']:
        s = sub[sub[a_col] == a_val]
        n = len(s)
        if n < min_n:
            print(f"    {a_name}={a_val:<8} (n={n:4d})  below min sample")
            continue
        bull = (s[b_col] == 'bullish').sum()
        bear = (s[b_col] == 'bearish').sum()
        print(f"    {a_name}={a_val:<8} (n={n:4d})  -> {b_name} bullish {safe_pct(bull,n):5.1f}%  "
              f"{b_name} bearish {safe_pct(bear,n):5.1f}%")
    print(f"\n  {b_name} direction -> {a_name} direction  (column %, 'the vice versa' -- diagnostic, not tradeable)")
    for b_val in ['bullish', 'bearish']:
        s = sub[sub[b_col] == b_val]
        n = len(s)
        if n < min_n:
            print(f"    {b_name}={b_val:<8} (n={n:4d})  below min sample")
            continue
        bull = (s[a_col] == 'bullish').sum()
        bear = (s[a_col] == 'bearish').sum()
        print(f"    {b_name}={b_val:<8} (n={n:4d})  -> {a_name} bullish {safe_pct(bull,n):5.1f}%  "
              f"{a_name} bearish {safe_pct(bear,n):5.1f}%")


def print_position_conditional(cond_col, cond_df, pos_col, base_rates):
    for cv in ['high', 'low']:
        subset = cond_df[cond_df[cond_col] == cv]
        n = len(subset)
        print(f"\n  {cond_col}={cv} (n={n}) -> {pos_col}")
        for bucket in buckets:
            count = (subset[pos_col] == bucket).sum()
            pct = safe_pct(count, n)
            b = base_rates[bucket]
            lift = round(pct / b, 2) if b > 0 else 0
            print(f"    {bucket:<15} {count:4d}  {pct:5.1f}%  (base {b:5.1f}%, lift {lift:.2f}x)")


edge_rows = []


def collect_edge(label, subset, base_bull_pct, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        return
    edge_rows.append({'label': label, 'n': s['n'], 'bull_pct': s['bull_pct'],
                       'bear_pct': s['bear_pct'], 'edge_vs_base': abs(s['bull_pct'] - base_bull_pct),
                       'bias': s['bias']})


def zscore_band(z):
    if pd.isna(z):
        return None
    if z < -1.5:
        return 'Very Small'
    if z < -0.5:
        return 'Small'
    if z < 0.5:
        return 'Average'
    if z < 1.5:
        return 'Large'
    return 'Very Large'


# ============================================================
# STAGE 1 -- LOAD THE 15y OHLCV CSV
# ============================================================
# Same loading convention as Orb_IB_positioning.py: Date + Time columns
# combined into a naive US/Eastern wall-clock DatetimeIndex. Everything
# downstream is naive-Eastern arithmetic, exactly like the v5 pipeline
# after its tz-strip.

def load_bars():
    print_header("STAGE 1 -- LOADING 15y 1-MIN OHLCV DATA")
    print(f"\n  Loading data from: {RAW_DATA_PATH}")

    df = pd.read_csv(RAW_DATA_PATH)
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
    df.set_index('Datetime', inplace=True)
    df = df.loc[START_DATE:END_DATE]
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='first')]

    rename = {'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close'}
    # accept a few common volume column spellings; absent -> price-only mode
    for vcand in ['Volume', 'volume', 'Vol', 'vol', 'VOLUME']:
        if vcand in df.columns:
            rename[vcand] = 'volume'
            break
    df = df.rename(columns=rename)

    has_volume = 'volume' in df.columns
    keep = ['open', 'high', 'low', 'close'] + (['volume'] if has_volume else [])
    df = df[keep]

    print(f"  Loaded {len(df):,} bars | {df.index.min()} -> {df.index.max()}")
    print(f"  Volume column present: {has_volume}"
          + ("" if has_volume else "  (VWAP sections will be skipped)"))
    return df, has_volume


# ============================================================
# STAGE 2 -- BUILD THE PER-DAY PRICE-STRUCTURE DATASET
# ============================================================

def build_dataset(bars, has_volume):
    print_header("STAGE 2 -- BUILDING PER-DAY PRICE-STRUCTURE DATASET")

    # ---- VWAP preprocessing (only meaningful with a volume column) ----
    if has_volume:
        bars = bars.copy()
        bars['typical_price'] = (bars['high'] + bars['low'] + bars['close']) / 3
        bars['pv'] = bars['typical_price'] * bars['volume']

        # Weekly anchor: resets Sunday 18:00 ET, runs through the week
        dow = bars.index.dayofweek  # Monday=0 ... Sunday=6
        days_since_sunday = (dow + 1) % 7
        candidate = pd.DatetimeIndex(bars.index.normalize()
                                     - pd.to_timedelta(days_since_sunday, unit='D')
                                     + pd.Timedelta(hours=18))
        needs_adjust = candidate > bars.index
        bars['week_start'] = pd.DatetimeIndex(
            np.where(needs_adjust, candidate - pd.Timedelta(days=7), candidate))

        bars['week_cum_pv'] = bars.groupby('week_start')['pv'].cumsum()
        bars['week_cum_vol'] = bars.groupby('week_start')['volume'].cumsum()
        bars['vwap_weekly'] = bars['week_cum_pv'] / bars['week_cum_vol']

        week_summary = bars.groupby('week_start').agg(wk_pv=('pv', 'sum'),
                                                      wk_vol=('volume', 'sum'))
        week_summary['wk_vwap'] = week_summary['wk_pv'] / week_summary['wk_vol']
        week_vwap_lookup = week_summary['wk_vwap'].to_dict()

        # Session anchor: resets 9:30 ET each day
        hm = bars.index.hour * 60 + bars.index.minute
        before_930 = hm < (9 * 60 + 30)
        sess_date = np.where(before_930,
                             bars.index.normalize() - pd.Timedelta(days=1),
                             bars.index.normalize())
        bars['session_start_930'] = (pd.DatetimeIndex(sess_date)
                                     + pd.Timedelta(hours=9, minutes=30))
        bars['day_cum_pv'] = bars.groupby('session_start_930')['pv'].cumsum()
        bars['day_cum_vol'] = bars.groupby('session_start_930')['volume'].cumsum()
        bars['vwap_session'] = bars['day_cum_pv'] / bars['day_cum_vol']

        print(f"  VWAP preprocessing done: {bars['week_start'].nunique()} weeks, "
              f"{bars['session_start_930'].nunique()} sessions")
    else:
        week_vwap_lookup = {}

    def week_start_for(ts):
        dow_ = ts.dayofweek
        days_since_sunday_ = (dow_ + 1) % 7
        cand = ts.normalize() - pd.Timedelta(days=int(days_since_sunday_)) + pd.Timedelta(hours=18)
        if cand > ts:
            cand -= pd.Timedelta(days=7)
        return cand

    # ---- Trading day enumeration (date = calendar date of RTH) ----
    shifted = (bars.index - pd.Timedelta(hours=7)).normalize()
    trading_days = sorted(pd.DatetimeIndex(shifted).unique())
    print(f"  {len(trading_days)} candidate trading days")

    records = []
    skipped = 0
    prev_range = None  # carried from the previous successfully-processed day

    for i, d in enumerate(trading_days[1:], 1):  # skip first (likely partial) day
        if i % 250 == 0:
            print(f"  Day {i} / {len(trading_days) - 1} ...")

        actual = pd.Timestamp(d) + pd.Timedelta(hours=7)

        asia_start = actual.replace(hour=18) - pd.Timedelta(days=1)
        asia_end = actual.replace(hour=2)
        lon_start = actual.replace(hour=2)
        lon_end = actual.replace(hour=8)

        ib_start = actual.replace(hour=9, minute=30)
        q1_end = actual.replace(hour=9, minute=45)   # ORB-15 end
        q2_end = actual.replace(hour=10, minute=0)   # ORB-30 end
        q3_end = actual.replace(hour=10, minute=15)
        ib_end = actual.replace(hour=10, minute=30)
        ny_end = actual.replace(hour=16, minute=0)

        asia_data = bars.loc[asia_start:asia_end]
        lon_data = bars.loc[lon_start:lon_end]
        orb15_data = bars.loc[ib_start:q1_end]
        orb30_data = bars.loc[ib_start:q2_end]
        q2_data = bars.loc[q1_end:q2_end]
        q3_data = bars.loc[q2_end:q3_end]
        q4_data = bars.loc[q3_end:ib_end]
        ib_data = bars.loc[ib_start:ib_end]
        post_ib_data = bars.loc[ib_end:ny_end]
        sh_after_orb15 = bars.loc[q2_end:ib_end]        # Q3+Q4, for ORB-15 curve
        full_day_data = bars.loc[asia_start:ny_end]     # seeds next day's prev_range

        if any(x.empty for x in [asia_data, lon_data, orb15_data, orb30_data,
                                  q2_data, q3_data, q4_data, ib_data, post_ib_data]):
            skipped += 1
            continue
        if len(orb15_data) < 10:  # need most of the 15 one-minute bars present
            skipped += 1
            continue

        # ---- ORB-15 / ORB-30 raw OHLC ----
        orb15_open, orb15_close = orb15_data['open'].iloc[0], orb15_data['close'].iloc[-1]
        orb15_high, orb15_low = orb15_data['high'].max(), orb15_data['low'].min()
        orb15_range = orb15_high - orb15_low
        if pd.isna([orb15_open, orb15_close, orb15_high, orb15_low]).any():
            skipped += 1
            continue

        orb30_open, orb30_close = orb30_data['open'].iloc[0], orb30_data['close'].iloc[-1]
        orb30_high, orb30_low = orb30_data['high'].max(), orb30_data['low'].min()
        orb30_range = orb30_high - orb30_low
        if pd.isna([orb30_open, orb30_close, orb30_high, orb30_low]).any():
            skipped += 1
            continue

        orb15_range_pct = (orb15_range / orb15_open) * 100 if orb15_open else np.nan
        orb30_range_pct = (orb30_range / orb30_open) * 100 if orb30_open else np.nan

        # ---- IB structure ----
        ib_open, ib_close = ib_data['open'].iloc[0], ib_data['close'].iloc[-1]
        ib_high, ib_low = ib_data['high'].max(), ib_data['low'].min()
        ib_range = ib_high - ib_low
        if pd.isna([ib_open, ib_close, ib_high, ib_low]).any() or ib_range == 0:
            skipped += 1
            continue

        ib_mid = (ib_high + ib_low) / 2
        ib_direction = 'bullish' if ib_close > ib_open else ('bearish' if ib_close < ib_open else None)
        if ib_direction is None:
            skipped += 1
            continue
        ib_close_position = 'above_midpoint' if ib_close > ib_mid else 'below_midpoint'
        ib_close_half_val = close_half(ib_close, ib_high, ib_low)
        ib_close_pct = position_in_range(ib_close, ib_high, ib_low)
        ib_close_zone = zone_label(ib_close_pct)
        ib_range_pct = (ib_range / ib_open) * 100 if ib_open else np.nan

        # ---- VWAP anchors ----
        if has_volume:
            current_week_start = week_start_for(actual)
            prev_week_start = current_week_start - pd.Timedelta(days=7)
            vwap_prevweek = week_vwap_lookup.get(prev_week_start, np.nan)
        else:
            vwap_prevweek = np.nan

        # ---- Premarket (6-9am) -- price structure (+ premarket VWAP) ----
        premarket_data = bars.loc[actual.replace(hour=6): actual.replace(hour=9)]
        if not premarket_data.empty:
            premarket_open = premarket_data['open'].iloc[0]
            premarket_close = premarket_data['close'].iloc[-1]
            premarket_high = premarket_data['high'].max()
            premarket_low = premarket_data['low'].min()
            premarket_range = premarket_high - premarket_low
            premarket_range_pct = (premarket_range / premarket_open) * 100 if premarket_open else np.nan
            premarket_direction = session_direction(premarket_open, premarket_close)
            premarket_close_half = close_half(premarket_close, premarket_high, premarket_low)
            if has_volume and premarket_data['volume'].sum() > 0:
                vwap_premarket = premarket_data['pv'].sum() / premarket_data['volume'].sum()
            else:
                vwap_premarket = np.nan
        else:
            premarket_open = premarket_close = premarket_high = premarket_low = np.nan
            premarket_range = premarket_range_pct = np.nan
            premarket_direction = None
            premarket_close_half = None
            vwap_premarket = np.nan

        if has_volume:
            vwap_weekly_at_orb15 = orb15_data['vwap_weekly'].iloc[-1]
            vwap_weekly_at_orb30 = orb30_data['vwap_weekly'].iloc[-1]
            vwap_weekly_at_ib = ib_data['vwap_weekly'].iloc[-1]
            vwap_session_at_orb15 = orb15_data['vwap_session'].iloc[-1]
            vwap_session_at_orb30 = orb30_data['vwap_session'].iloc[-1]
            vwap_session_at_ib = ib_data['vwap_session'].iloc[-1]
        else:
            vwap_weekly_at_orb15 = vwap_weekly_at_orb30 = vwap_weekly_at_ib = np.nan
            vwap_session_at_orb15 = vwap_session_at_orb30 = vwap_session_at_ib = np.nan

        orb15_vs_vwap_weekly = vs_level(orb15_close, vwap_weekly_at_orb15)
        orb30_vs_vwap_weekly = vs_level(orb30_close, vwap_weekly_at_orb30)
        ib_vs_vwap_weekly = vs_level(ib_close, vwap_weekly_at_ib)

        orb15_vs_vwap_prevweek = vs_level(orb15_close, vwap_prevweek)
        orb30_vs_vwap_prevweek = vs_level(orb30_close, vwap_prevweek)
        ib_vs_vwap_prevweek = vs_level(ib_close, vwap_prevweek)

        orb15_vs_vwap_premarket = vs_level(orb15_close, vwap_premarket)
        orb30_vs_vwap_premarket = vs_level(orb30_close, vwap_premarket)
        ib_vs_vwap_premarket = vs_level(ib_close, vwap_premarket)

        orb15_vs_vwap_session = vs_level(orb15_close, vwap_session_at_orb15)
        orb30_vs_vwap_session = vs_level(orb30_close, vwap_session_at_orb30)
        ib_vs_vwap_session = vs_level(ib_close, vwap_session_at_ib)

        # ---- ORB meaningfulness vs. previous day's full-session range ----
        orb15_meaningful = (prev_range is not None and prev_range > 0 and
                            orb15_range >= prev_range * MIN_ORB_RANGE_FRACTION)
        orb30_meaningful = (prev_range is not None and prev_range > 0 and
                            orb30_range >= prev_range * MIN_ORB_RANGE_FRACTION)

        orb15_close_half_val = close_half(orb15_close, orb15_high, orb15_low) if orb15_meaningful else None
        orb15_direction = session_direction(orb15_open, orb15_close)
        orb15_mid = (orb15_high + orb15_low) / 2
        orb15_pos_in_ib = position_in_range(orb15_mid, ib_high, ib_low)
        orb15_pos_bucket = position_bucket(orb15_pos_in_ib)

        orb30_close_half_val = close_half(orb30_close, orb30_high, orb30_low) if orb30_meaningful else None
        orb30_direction = session_direction(orb30_open, orb30_close)
        orb30_mid = (orb30_high + orb30_low) / 2
        orb30_pos_in_ib = position_in_range(orb30_mid, ib_high, ib_low)
        orb30_pos_bucket = position_bucket(orb30_pos_in_ib)

        # ---- ORB-15 / ORB-30 internal extreme-first (live-safe) ----
        orb15_internal_extreme_first = which_extreme_first(orb15_data, orb15_high, orb15_low)
        orb30_internal_extreme_first = which_extreme_first(orb30_data, orb30_high, orb30_low)

        # ---- IB extreme first (look-ahead if extreme sets in Q3/Q4) ----
        ib_extreme_first = which_extreme_first(ib_data, ib_high, ib_low)

        # ---- SECOND bullish/bearish definition: extreme-first + close-half
        #      bullish = low first + upper close ; bearish = high first + lower close
        #      (orb15/orb30 use their own internal, live-safe extreme-first;
        #      IB uses its final extreme-first, which carries look-ahead the
        #      same way ib_extreme_first itself does) ----
        orb15_direction2 = extreme_close_direction(orb15_internal_extreme_first, orb15_close_half_val)
        orb30_direction2 = extreme_close_direction(orb30_internal_extreme_first, orb30_close_half_val)
        ib_direction2 = extreme_close_direction(ib_extreme_first, ib_close_half_val)

        # ---- IB quarter / curve type ----
        ib_high_time = ib_data[ib_data['high'] == ib_high].index[0]
        ib_low_time = ib_data[ib_data['low'] == ib_low].index[0]
        ib_high_q = which_quarter(ib_high_time, q1_end, q2_end, q3_end)
        ib_low_q = which_quarter(ib_low_time, q1_end, q2_end, q3_end)
        ib_high_half = 1 if ib_high_time < q2_end else 2
        ib_low_half = 1 if ib_low_time < q2_end else 2
        if ib_high_half != ib_low_half:
            ib_curve_type = 'high_first' if ib_high_half == 1 else 'low_first'
        elif ib_high_half == 1:
            ib_curve_type = 'both_first_half'
        else:
            ib_curve_type = 'both_second_half'

        # ---- Curve results ----
        orb15_curve = compute_curve_result(
            q2_data['high'].max(), q2_data['low'].min(),
            orb15_high, orb15_low,
            sh_after_orb15['high'].max(), sh_after_orb15['low'].min(),
        )
        orb30_curve_within_ib = compute_curve_result(
            q3_data['high'].max(), q3_data['low'].min(),
            orb30_high, orb30_low,
            q4_data['high'].max(), q4_data['low'].min(),
        )
        orb30_curve_post_ib = compute_curve_result(
            q3_data['high'].max(), q3_data['low'].min(),
            orb30_high, orb30_low,
            post_ib_data['high'].max(), post_ib_data['low'].min(),
        )

        # ---- Chain agreement flags ----
        orb15_ib_close_agree = (orb15_close_half_val is not None and
                                 orb15_close_half_val == ib_close_half_val)
        orb30_ib_close_agree = (orb30_close_half_val is not None and
                                 orb30_close_half_val == ib_close_half_val)
        extreme_close_agree = None
        if ib_extreme_first in ('high', 'low'):
            expected = 'lower' if ib_extreme_first == 'high' else 'upper'
            extreme_close_agree = (ib_close_half_val == expected)

        # ---- Session context ----
        asia_open, asia_close = asia_data['open'].iloc[0], asia_data['close'].iloc[-1]
        asia_high, asia_low = asia_data['high'].max(), asia_data['low'].min()
        asia_dir = session_direction(asia_open, asia_close)

        lon_open, lon_close = lon_data['open'].iloc[0], lon_data['close'].iloc[-1]
        lon_high, lon_low = lon_data['high'].max(), lon_data['low'].min()
        lon_dir = session_direction(lon_open, lon_close)

        if pd.isna([asia_high, asia_low, lon_high, lon_low]).any():
            skipped += 1
            continue

        asia_london_pattern = classify_asia_london(lon_high, lon_low, asia_high, asia_low)

        lon_mid = (lon_high + lon_low) / 2
        overnight_high = max(asia_high, lon_high)
        overnight_low = min(asia_low, lon_low)
        overnight_mid = (overnight_high + overnight_low) / 2

        ny_8am_data = bars.loc[lon_end: lon_end + pd.Timedelta(minutes=1)]
        ny_8am_open = ny_8am_data['open'].iloc[0] if not ny_8am_data.empty else ib_open

        ny_open_vs_london_mid = 'above' if ny_8am_open > lon_mid else 'below'
        ny_open_vs_overnight_mid = 'above' if ny_8am_open > overnight_mid else 'below'

        orb15_vs_london_mid = 'above' if orb15_close > lon_mid else 'below'
        orb30_vs_london_mid = 'above' if orb30_close > lon_mid else 'below'
        ib_vs_london_mid = 'above' if ib_close > lon_mid else 'below'

        # ---- Post-IB outcome / break tracking ----
        post_ib_close = post_ib_data['close'].iloc[-1]
        post_ib_dir = session_direction(ib_close, post_ib_close)

        one_hour_post_ib = bars.loc[ib_end:ib_end + pd.Timedelta(hours=1)]
        hb_1h, hb_t_1h, lb_1h, lb_t_1h = find_first_break(one_hour_post_ib, ib_high, ib_low)
        hb_ses, hb_t_ses, lb_ses, lb_t_ses = find_first_break(post_ib_data, ib_high, ib_low)

        first_break_dir = None
        for _, row in post_ib_data.iterrows():
            bh = row['high'] > ib_high
            bl = row['low'] < ib_low
            if bh and bl:
                first_break_dir = 'simultaneous'
                break
            elif bh:
                first_break_dir = 'high'
                break
            elif bl:
                first_break_dir = 'low'
                break

        trade_type = compute_trade_type(ib_extreme_first, ib_close_half_val)

        # ---- Structural score (10 signals -- order-flow score removed) ----
        struct_signals = [
            score_dir(asia_dir),
            score_dir(lon_dir),
            score_pattern(asia_london_pattern),
            score_close_half(orb15_close_half_val),
            score_position(orb15_pos_bucket),
            score_close_half(orb30_close_half_val),
            score_position(orb30_pos_bucket),
            score_extreme_first(ib_extreme_first),
            score_close_half(ib_close_half_val),
            score_ny_vs_mid(ny_open_vs_london_mid),
        ]
        struct_bias, struct_bull, struct_bear, struct_agreement = aggregate(
            struct_signals, MIN_STRUCTURAL_AGREEMENT)
        structural_score_label = struct_bias if struct_agreement else 'no_edge'

        record = {
            'date': actual.date(),
            'prev_range': prev_range,

            # session context
            'asia_direction': asia_dir, 'london_direction': lon_dir,
            'asia_london_pattern': asia_london_pattern,
            'ny_open_vs_london_mid': ny_open_vs_london_mid,
            'ny_open_vs_overnight_mid': ny_open_vs_overnight_mid,
            'lon_high': lon_high, 'lon_low': lon_low, 'lon_mid': lon_mid,
            'asia_high': asia_high, 'asia_low': asia_low,
            'overnight_high': overnight_high, 'overnight_low': overnight_low,
            'overnight_mid': overnight_mid,

            # VWAP anchors
            'vwap_prevweek': vwap_prevweek, 'vwap_premarket': vwap_premarket,
            'vwap_weekly_at_orb15': vwap_weekly_at_orb15, 'vwap_weekly_at_orb30': vwap_weekly_at_orb30,
            'vwap_weekly_at_ib': vwap_weekly_at_ib,
            'vwap_session_at_orb15': vwap_session_at_orb15, 'vwap_session_at_orb30': vwap_session_at_orb30,
            'vwap_session_at_ib': vwap_session_at_ib,
            'orb15_vs_vwap_weekly': orb15_vs_vwap_weekly, 'orb30_vs_vwap_weekly': orb30_vs_vwap_weekly,
            'ib_vs_vwap_weekly': ib_vs_vwap_weekly,
            'orb15_vs_vwap_prevweek': orb15_vs_vwap_prevweek, 'orb30_vs_vwap_prevweek': orb30_vs_vwap_prevweek,
            'ib_vs_vwap_prevweek': ib_vs_vwap_prevweek,
            'orb15_vs_vwap_premarket': orb15_vs_vwap_premarket, 'orb30_vs_vwap_premarket': orb30_vs_vwap_premarket,
            'ib_vs_vwap_premarket': ib_vs_vwap_premarket,
            'orb15_vs_vwap_session': orb15_vs_vwap_session, 'orb30_vs_vwap_session': orb30_vs_vwap_session,
            'ib_vs_vwap_session': ib_vs_vwap_session,

            # 6-9am premarket -- price structure only
            'premarket_open': premarket_open, 'premarket_close': premarket_close,
            'premarket_high': premarket_high, 'premarket_low': premarket_low,
            'premarket_range': premarket_range, 'premarket_range_pct': premarket_range_pct,
            'premarket_direction': premarket_direction, 'premarket_close_half': premarket_close_half,

            # ORB-15
            'orb15_open': orb15_open, 'orb15_close': orb15_close,
            'orb15_high': orb15_high, 'orb15_low': orb15_low,
            'orb15_vs_london_mid': orb15_vs_london_mid,
            'orb15_range': orb15_range, 'orb15_range_pct': orb15_range_pct,
            'orb15_meaningful': orb15_meaningful,
            'orb15_direction': orb15_direction, 'orb15_close_half': orb15_close_half_val,
            'orb15_pos_in_ib': orb15_pos_in_ib, 'orb15_pos_bucket': orb15_pos_bucket,
            'orb15_curve_result': orb15_curve,
            'orb15_ib_close_agree': orb15_ib_close_agree,
            'orb15_internal_extreme_first': orb15_internal_extreme_first,
            'orb15_direction2': orb15_direction2,

            # ORB-30
            'orb30_open': orb30_open, 'orb30_close': orb30_close,
            'orb30_high': orb30_high, 'orb30_low': orb30_low,
            'orb30_vs_london_mid': orb30_vs_london_mid,
            'orb30_range': orb30_range, 'orb30_range_pct': orb30_range_pct,
            'orb30_meaningful': orb30_meaningful,
            'orb30_direction': orb30_direction, 'orb30_close_half': orb30_close_half_val,
            'orb30_pos_in_ib': orb30_pos_in_ib, 'orb30_pos_bucket': orb30_pos_bucket,
            'orb30_curve_result_within_ib': orb30_curve_within_ib,
            'orb30_curve_result_post_ib': orb30_curve_post_ib,
            'orb30_ib_close_agree': orb30_ib_close_agree,
            'orb30_internal_extreme_first': orb30_internal_extreme_first,
            'orb30_direction2': orb30_direction2,

            # IB structure
            'ib_open': ib_open, 'ib_close': ib_close, 'ib_high': ib_high, 'ib_low': ib_low,
            'ib_range': ib_range, 'ib_range_pct': ib_range_pct, 'ib_mid': ib_mid,
            'ib_direction': ib_direction, 'ib_close_position': ib_close_position,
            'ib_close_half': ib_close_half_val, 'ib_close_pct': ib_close_pct,
            'ib_close_zone': ib_close_zone,
            'ib_vs_london_mid': ib_vs_london_mid,
            'ib_extreme_first': ib_extreme_first,
            'ib_direction2': ib_direction2,
            'ib_curve_type': ib_curve_type,
            'ib_high_quarter': ib_high_q, 'ib_low_quarter': ib_low_q,
            'extreme_close_agree': extreme_close_agree,

            # breaks / outcome
            'ib_high_broken_1h': hb_1h, 'ib_low_broken_1h': lb_1h,
            'ib_high_broken_session': hb_ses, 'ib_low_broken_session': lb_ses,
            'first_break_dir': first_break_dir,
            'post_ib_direction': post_ib_dir,

            # scoring
            'structural_bull_count': struct_bull, 'structural_bear_count': struct_bear,
            'structural_score_label': structural_score_label,
            'trade_type': trade_type,
        }
        records.append(record)

        # carry this day's full-session range forward as next day's prev_range
        prev_range = full_day_data['high'].max() - full_day_data['low'].min()

    rdf = pd.DataFrame(records)
    print(f"\nBuilt {len(rdf):,} daily records | skipped {skipped}")

    # ---- Size (range %) z-scores, percentiles, bands ----
    SIZE_COLS = ['orb15_range_pct', 'orb30_range_pct', 'ib_range_pct', 'premarket_range_pct']
    for col in SIZE_COLS:
        rdf[f'{col}_zscore'] = scipy_stats.zscore(rdf[col], nan_policy='omit')
        rdf[f'{col}_pctile'] = rdf[col].rank(pct=True) * 100
        rdf[f'{col}_band'] = rdf[f'{col}_zscore'].apply(zscore_band)

    print("\nSize band distribution (IB range %):")
    print(rdf['ib_range_pct_band'].value_counts())
    print("\nStructural score distribution:")
    print(rdf['structural_score_label'].value_counts())

    rdf.to_csv(OUTPUT_DATASET_PATH, index=False)
    print(f"\nSaved unified dataset -> {OUTPUT_DATASET_PATH}")
    return rdf


# ============================================================
# STAGE 3 -- ANALYSIS
# ============================================================

def run_analysis(rdf, has_volume):
    total_days = len(rdf)

    BASE_STATS = outcome_stats(rdf, min_n=1)
    BASE_BULL_PCT = BASE_STATS['bull_pct'] if BASE_STATS else 50.0
    print(f"\n  Baseline post-IB direction: Bull {BASE_BULL_PCT:.1f}%  (n={BASE_STATS['n'] if BASE_STATS else 0})")

    IB_BASE_STATS = outcome_stats(rdf, direction_col='ib_direction', min_n=1)
    BASE_IB_BULL_PCT = IB_BASE_STATS['bull_pct'] if IB_BASE_STATS else 50.0
    print(f"  Baseline IB direction: Bull {BASE_IB_BULL_PCT:.1f}%  (n={IB_BASE_STATS['n'] if IB_BASE_STATS else 0})")

    ef = rdf[rdf['ib_extreme_first'].isin(['high', 'low'])]
    fb = rdf[rdf['first_break_dir'].isin(['high', 'low'])]
    meaningful15 = rdf[rdf['orb15_meaningful'] == True]
    meaningful30 = rdf[rdf['orb30_meaningful'] == True]

    # ========================================================
    # ITEM 1 -- ORB DIRECTION <-> IB DIRECTION
    # ========================================================
    print_header("ITEM 1 -- IF THE ORB IS BULLISH, IS THE IB BULLISH? (AND VICE VERSA)")
    direction_concordance(rdf, 'orb15_direction', 'ib_direction', 'ORB-15', 'IB')
    direction_concordance(rdf, 'orb30_direction', 'ib_direction', 'ORB-30', 'IB')

    # ========================================================
    # ITEM 1B -- SECOND BULLISH/BEARISH DEFINITION
    # ========================================================
    # direction  (Item 1 above) = open vs close, ignores where in the range
    #            the close sits or which extreme printed first.
    # direction2 (here)         = bullish only if the low formed first AND
    #            the close sits in the upper half; bearish only if the high
    #            formed first AND the close sits in the lower half. Any other
    #            combination (extreme-first/close-half pointing the same way,
    #            simultaneous extremes, or a non-meaningful/absent ORB) is
    #            neither and comes through as None -- so n here will be
    #            noticeably smaller than Item 1's n, by construction.
    print_header("ITEM 1B -- SECOND DEFINITION: LOW/HIGH-FIRST + CLOSE-HALF")
    print("  bullish = low formed first AND close in upper half")
    print("  bearish = high formed first AND close in lower half")
    print("  (orb15/orb30 use their own internal live-safe extreme-first;")
    print("   IB uses its final extreme-first, same look-ahead caveat as ib_extreme_first)\n")

    for label, col in [('ORB-15', 'orb15_direction2'), ('ORB-30', 'orb30_direction2'),
                       ('IB', 'ib_direction2')]:
        n_valid = rdf[col].notna().sum()
        print_subheader(f"{label} direction2 -- distribution (n valid = {n_valid} / {total_days})")
        for val in ['bullish', 'bearish']:
            print_row(f"{label} direction2={val}", (rdf[col] == val).sum(), total_days)

    print_subheader("ORB-15 direction2 -> IB direction (concordance)")
    direction_concordance(rdf, 'orb15_direction2', 'ib_direction', 'ORB-15_dir2', 'IB')

    print_subheader("ORB-30 direction2 -> IB direction (concordance)")
    direction_concordance(rdf, 'orb30_direction2', 'ib_direction', 'ORB-30_dir2', 'IB')

    print_subheader("IB direction2 -> Post-IB direction / break rates")
    for val in ['bullish', 'bearish']:
        subset = rdf[rdf['ib_direction2'] == val]
        print_outcome(f"ib_direction2={val}", subset)
        print_break(f"ib_direction2={val}", subset)
        collect_edge(f"ib_direction2={val}", subset, BASE_BULL_PCT)

    print_subheader("ORB-15 / ORB-30 direction2 -> IB direction (as an outcome, live-safe)")
    for label, col in [('ORB-15', 'orb15_direction2'), ('ORB-30', 'orb30_direction2')]:
        for val in ['bullish', 'bearish']:
            subset = rdf[rdf[col] == val]
            print_outcome(f"{label}_direction2={val}", subset, direction_col='ib_direction')
            collect_edge(f"{label}_direction2={val}_vs_ib", subset, BASE_IB_BULL_PCT,
                        direction_col='ib_direction')

    print_subheader("Does direction2 agree with the open-vs-close direction (Item 1)?")
    for label, d2_col, d1_col in [('ORB-15', 'orb15_direction2', 'orb15_direction'),
                                  ('ORB-30', 'orb30_direction2', 'orb30_direction'),
                                  ('IB', 'ib_direction2', 'ib_direction')]:
        sub = rdf[rdf[d2_col].notna() & rdf[d1_col].notna()]
        n = len(sub)
        if n == 0:
            print(f"    {label}: no valid rows")
            continue
        agree = (sub[d2_col] == sub[d1_col]).sum()
        print(f"    {label}: direction2 agrees with open-vs-close direction on "
              f"{safe_pct(agree, n):5.1f}%  (n={n})")

    # ========================================================
    # ITEM 2 -- BREAK RATES BY IB CLOSE HALF AND EXTREME-FIRST
    # ========================================================
    print_header("ITEM 2 -- BREAK RATES BY IB CLOSE HALF AND EXTREME-FIRST")

    print_subheader("Break rates by IB close half (marginal)")
    for ch in ['upper', 'lower']:
        print_break(f"ib_close={ch}", rdf[rdf['ib_close_half'] == ch])

    print_subheader("Break rates by IB extreme-first (marginal)")
    for ext in ['high', 'low']:
        print_break(f"extreme_first={ext}", rdf[rdf['ib_extreme_first'] == ext])

    print_subheader("Break rates by IB close half x extreme-first (joint, 2x2)")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['ib_close_half'] == ch) & (rdf['ib_extreme_first'] == ext)]
            print_break(f"ib_close={ch}+extreme={ext}", subset)
            collect_edge(f"ib_close={ch}+extreme={ext}", subset, BASE_BULL_PCT)

    # ========================================================
    # ITEM 3 -- ORB POSITION IN IB, CONDITIONED ON EXTREME/BREAK-FIRST
    # ========================================================
    print_header("ITEM 3 -- ORB POSITION IN IB, CONDITIONED ON EXTREME-FIRST AND BREAK-FIRST")
    print("  (Note: this is the ORBs' position within the IB, not the IB's own structure.)\n")

    base_ef_b15 = {b: safe_pct((ef['orb15_pos_bucket'] == b).sum(), len(ef)) for b in buckets}
    base_ef_b30 = {b: safe_pct((ef['orb30_pos_bucket'] == b).sum(), len(ef)) for b in buckets}
    base_fb_b15 = {b: safe_pct((fb['orb15_pos_bucket'] == b).sum(), len(fb)) for b in buckets}
    base_fb_b30 = {b: safe_pct((fb['orb30_pos_bucket'] == b).sum(), len(fb)) for b in buckets}

    print_subheader(f"ORB-15 position, conditioned on which IB extreme formed FIRST (n={len(ef)})")
    print_position_conditional('ib_extreme_first', ef, 'orb15_pos_bucket', base_ef_b15)

    print_subheader(f"ORB-30 position, conditioned on which IB extreme formed FIRST (n={len(ef)})")
    print_position_conditional('ib_extreme_first', ef, 'orb30_pos_bucket', base_ef_b30)

    print_subheader(f"ORB-15 position, conditioned on which direction BROKE FIRST post-IB (n={len(fb)})")
    print_position_conditional('first_break_dir', fb, 'orb15_pos_bucket', base_fb_b15)

    print_subheader(f"ORB-30 position, conditioned on which direction BROKE FIRST post-IB (n={len(fb)})")
    print_position_conditional('first_break_dir', fb, 'orb30_pos_bucket', base_fb_b30)

    # ========================================================
    # TRADING FRAMEWORK -- 10:00-10:30 DECISION WINDOW
    # ========================================================
    print_header("TRADING FRAMEWORK -- 10:00-10:30 DECISION WINDOW")
    print("  Given ORB close-half + which extreme formed first, what % of the time")
    print("  does the IB itself end up bullish vs. bearish?\n")
    print("  ib_extreme_first uses the IB's FINAL high/low -- has look-ahead if the")
    print("  true extreme prints in Q3/Q4. orb30_internal_extreme_first is fully known")
    print("  at 10:00am (ORB-30's own high/low order) -- this is the live-safe version.\n")

    for label, mdf, ch_col in [('ORB-15', meaningful15, 'orb15_close_half'),
                               ('ORB-30', meaningful30, 'orb30_close_half')]:
        print_subheader(f"{label} close-half x IB extreme-first (LOOK-AHEAD) -> IB direction")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = mdf[(mdf[ch_col] == ch) & (mdf['ib_extreme_first'] == ext)]
                print_outcome(f"{label}={ch}+ib_extreme_first={ext}", subset, direction_col='ib_direction')
                collect_edge(f"{label}={ch}+ib_extreme_first={ext}", subset, BASE_BULL_PCT)

    print_subheader("ORB-30 close-half x ORB-30's OWN extreme-first (LIVE-SAFE) -> IB direction")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = meaningful30[(meaningful30['orb30_close_half'] == ch) &
                                  (meaningful30['orb30_internal_extreme_first'] == ext)]
            print_outcome(f"ORB-30={ch}+orb30_internal_extreme_first={ext}", subset, direction_col='ib_direction')
            collect_edge(f"ORB-30={ch}+orb30_internal_extreme_first={ext}", subset, BASE_BULL_PCT)

    print_subheader("ORB-15 close-half x ORB-15's OWN extreme-first (LIVE-SAFE at 9:45am) -> IB direction")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = meaningful15[(meaningful15['orb15_close_half'] == ch) &
                                  (meaningful15['orb15_internal_extreme_first'] == ext)]
            print_outcome(f"ORB-15={ch}+orb15_internal_extreme_first={ext}", subset, direction_col='ib_direction')
            collect_edge(f"ORB-15={ch}+orb15_internal_extreme_first={ext}", subset, BASE_BULL_PCT)

    # ========================================================
    # ITEM 4 -- ORB / IB POSITION RELATIVE TO LONDON MIDPOINT
    # ========================================================
    print_header("ITEM 4 -- POSITION OF ORBS / IB RELATIVE TO THE LONDON MIDPOINT")

    for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('ORB-30', 'orb30_vs_london_mid'),
                       ('IB', 'ib_vs_london_mid')]:
        print_subheader(f"{label} close vs. London midpoint")
        for pos in ['above', 'below']:
            print_row(f"{label} closes {pos} London mid", (rdf[col] == pos).sum(), total_days)
        print()
        for pos in ['above', 'below']:
            subset = rdf[rdf[col] == pos]
            print_outcome(f"{label} {pos} London mid -> Post-IB", subset)
            print_break(f"{label} {pos} London mid", subset)
            collect_edge(f"{label}_vs_london_mid={pos}", subset, BASE_BULL_PCT)

    # ========================================================
    # ITEM 5 -- ORB / IB POSITION BY SESSION STRUCTURAL PATTERN
    # ========================================================
    print_header("ITEM 5 -- ORB/IB POSITION BY ASIA/LONDON STRUCTURAL PATTERN")

    for pat in ['london_engulfs_asia', 'asia_engulfs_london',
                'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
        pat_sub = rdf[rdf['asia_london_pattern'] == pat]
        n_pat = len(pat_sub)
        print_subheader(f"{pat}  (n={n_pat})")
        if n_pat < MIN_DISPLAY_SAMPLE:
            print("    below min sample")
            continue
        for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('ORB-30', 'orb30_vs_london_mid'),
                           ('IB', 'ib_vs_london_mid')]:
            above = (pat_sub[col] == 'above').sum()
            print(f"    {label} closes above London mid: {safe_pct(above, n_pat):5.1f}%  ({above}/{n_pat})")
        print()
        print_outcome(f"{pat} -> Post-IB", pat_sub)
        print_break(pat, pat_sub)
        collect_edge(f"asia_london_pattern={pat}", pat_sub, BASE_BULL_PCT)

        for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('IB', 'ib_vs_london_mid')]:
            for pos in ['above', 'below']:
                sub2 = pat_sub[pat_sub[col] == pos]
                print_outcome(f"  {pat[:24]} + {label}={pos}", sub2)
                collect_edge(f"pattern={pat[:24]}+{label}_vs_lonmid={pos}", sub2, BASE_BULL_PCT)

    # ========================================================
    # SIZE (was Stage 4 Part A) -- IS THE ORB/IB RANGE LARGER/SMALLER THAN NORMAL?
    # ========================================================
    print_header("SIZE -- IS THE ORB/IB RANGE LARGER OR SMALLER THAN NORMAL?")

    for label, band_col in [('ORB-15', 'orb15_range_pct_band'), ('ORB-30', 'orb30_range_pct_band'),
                            ('IB', 'ib_range_pct_band')]:
        print_subheader(f"{label} range size-band distribution + outcomes")
        for band in BAND_ORDER:
            print_row(f"{label} = {band}", (rdf[band_col] == band).sum(), total_days)
        print()
        for band in BAND_ORDER:
            subset = rdf[rdf[band_col] == band]
            print_outcome(f"{label}={band} -> Post-IB", subset)
            print_break(f"{label}={band}", subset)
            collect_edge(f"{label}_size={band}", subset, BASE_BULL_PCT)

    print_subheader("IB size band x close-half x extreme-first (does size change the earlier pattern?)")
    for band in ['Small', 'Average', 'Large']:
        band_sub = rdf[rdf['ib_range_pct_band'] == band]
        print(f"\n  IB size = {band} (n={len(band_sub)})")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
                print_break(f"    ib={ch}+ext={ext}", subset)
                collect_edge(f"size={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)

    print_subheader("Does ORB-30 SIZE sharpen the live-safe ORB-30 signal?")
    print("  Baseline: ORB-30 close-half x its own internal extreme-first -> IB direction\n")
    for band in ['Small', 'Average', 'Large']:
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['orb30_range_pct_band'] == band) & (rdf['orb30_close_half'] == ch) &
                             (rdf['orb30_internal_extreme_first'] == ext)]
                label = f"orb30_size={band}+ch={ch}+ext={ext}"
                print_outcome(label, subset, direction_col='ib_direction')
                collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

    print_subheader("Baseline for comparison (no size factor)")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
            print_outcome(f"orb30={ch}+ext={ext}  (no size)", subset, direction_col='ib_direction')

    # ========================================================
    # VWAP ANCHORS (was Stage 5) -- only meaningful with a volume column
    # ========================================================
    ANCHORS = [
        ('weekly',    'Weekly (Sun 18:00 anchor, running)'),
        ('session',   'Session (9:30 anchor, running -- classic day-VWAP)'),
        ('prevweek',  "Previous week's VWAP (static)"),
        ('premarket', '6-9am premarket VWAP (static)'),
    ]
    PRICE_POINTS = [('ORB-15', 'orb15'), ('ORB-30', 'orb30'), ('IB', 'ib')]

    if has_volume:
        print_header("VWAP -- PART A: POSITION VS. EACH VWAP ANCHOR")

        for anchor, anchor_label in ANCHORS:
            print_subheader(anchor_label)
            for label, prefix in PRICE_POINTS:
                col = f'{prefix}_vs_vwap_{anchor}'
                above = (rdf[col] == 'above').sum()
                below = (rdf[col] == 'below').sum()
                valid = above + below
                print(f"\n  {label} vs {anchor} VWAP  (n valid = {valid})")
                for pos in ['above', 'below']:
                    subset = rdf[rdf[col] == pos]
                    print_outcome(f"    {label} {pos}", subset)
                    print_break(f"    {label} {pos}", subset)
                    collect_edge(f"{label}_vs_vwap_{anchor}={pos}", subset, BASE_BULL_PCT)

        print_header("VWAP -- PART B: VWAP POSITION BY ASIA/LONDON STRUCTURAL PATTERN")
        print("  (weekly and session anchors only)\n")
        for pat in ['london_engulfs_asia', 'asia_engulfs_london',
                    'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
            pat_sub = rdf[rdf['asia_london_pattern'] == pat]
            n_pat = len(pat_sub)
            print_subheader(f"{pat}  (n={n_pat})")
            if n_pat < MIN_DISPLAY_SAMPLE:
                print("    below min sample")
                continue
            for anchor in ['weekly', 'session']:
                col = f'ib_vs_vwap_{anchor}'
                above = (pat_sub[col] == 'above').sum()
                valid = (pat_sub[col].isin(['above', 'below'])).sum()
                print(f"    IB above {anchor} VWAP: {safe_pct(above, valid):5.1f}%  ({above}/{valid})")
            print()
            for anchor in ['weekly', 'session']:
                col = f'ib_vs_vwap_{anchor}'
                for pos in ['above', 'below']:
                    sub2 = pat_sub[pat_sub[col] == pos]
                    print_outcome(f"  {pat[:22]} + IB {anchor}={pos}", sub2)
                    collect_edge(f"pattern={pat[:22]}+ib_vwap_{anchor}={pos}", sub2, BASE_BULL_PCT)

        print_header("VWAP -- PART C: DOES VWAP SHARPEN THE ORB-30 TRADING-FRAMEWORK SIGNAL?")
        print("  Baseline: ORB-30 close-half x its own internal extreme-first")
        print("  -> directional read on IB direction, live-safe at 10:00am.\n")
        for anchor in ['weekly', 'session']:
            vwap_col = f'orb30_vs_vwap_{anchor}'
            print_subheader(f"ORB-30 close-half x internal extreme-first x vs. {anchor} VWAP -> IB direction")
            for ch in ['upper', 'lower']:
                for ext in ['high', 'low']:
                    for vwap_pos in ['above', 'below']:
                        subset = rdf[(rdf['orb30_close_half'] == ch) &
                                     (rdf['orb30_internal_extreme_first'] == ext) &
                                     (rdf[vwap_col] == vwap_pos)]
                        label = f"orb30={ch}+ext={ext}+vwap_{anchor}={vwap_pos}"
                        print_outcome(label, subset, direction_col='ib_direction')
                        collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

        print_subheader("For comparison: the two-factor baseline without VWAP")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
                print_outcome(f"orb30={ch}+ext={ext}  (no VWAP)", subset, direction_col='ib_direction')
    else:
        print_header("VWAP SECTIONS SKIPPED -- no volume column in the source CSV")

    # ========================================================
    # PREMARKET (was Stage 6, price parts only)
    # ========================================================
    print_header("PREMARKET (6-9AM) -- SIZE: LARGER OR SMALLER THAN NORMAL?")

    print_subheader("Premarket range size-band distribution + outcomes")
    for band in BAND_ORDER:
        print_row(f"premarket = {band}", (rdf['premarket_range_pct_band'] == band).sum(), total_days)
    print()
    for band in BAND_ORDER:
        subset = rdf[rdf['premarket_range_pct_band'] == band]
        print_outcome(f"premarket={band} -> Post-IB", subset)
        print_break(f"premarket={band}", subset)
        collect_edge(f"premarket_size={band}", subset, BASE_BULL_PCT)

    print_subheader("Premarket size band x IB close-half x IB extreme-first")
    for band in ['Small', 'Average', 'Large']:
        band_sub = rdf[rdf['premarket_range_pct_band'] == band]
        print(f"\n  premarket size = {band} (n={len(band_sub)})")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
                print_break(f"    ib={ch}+ext={ext}", subset)
                collect_edge(f"premarket_size={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)

    print_header("PREMARKET -- DOES PREMARKET DIRECTION CONCORD WITH IB DIRECTION? (AND VICE VERSA)")

    sub = rdf[rdf['premarket_direction'].notna() & rdf['ib_direction'].notna()]
    bb = ((sub['premarket_direction'] == 'bullish') & (sub['ib_direction'] == 'bullish')).sum()
    bear_b = ((sub['premarket_direction'] == 'bullish') & (sub['ib_direction'] == 'bearish')).sum()
    b_bear = ((sub['premarket_direction'] == 'bearish') & (sub['ib_direction'] == 'bullish')).sum()
    bearbear = ((sub['premarket_direction'] == 'bearish') & (sub['ib_direction'] == 'bearish')).sum()
    print(f"\n  Raw joint counts (n={len(sub)}):")
    print(f"  {'':<20}{'IB bull':>12}{'IB bear':>12}{'Total':>10}")
    print(f"  {'premarket bull':<20}{bb:>12}{bear_b:>12}{bb+bear_b:>10}")
    print(f"  {'premarket bear':<20}{b_bear:>12}{bearbear:>12}{b_bear+bearbear:>10}")
    print(f"  {'Total':<20}{bb+b_bear:>12}{bear_b+bearbear:>12}{len(sub):>10}")

    print(f"\n  premarket direction -> IB direction (row %, the tradeable direction, known by 9:00am)")
    for val in ['bullish', 'bearish']:
        s = sub[sub['premarket_direction'] == val]
        n = len(s)
        bull = (s['ib_direction'] == 'bullish').sum()
        bear = (s['ib_direction'] == 'bearish').sum()
        print(f"    premarket={val:<8} (n={n:4d})  -> IB bullish {safe_pct(bull,n):5.1f}%  "
              f"IB bearish {safe_pct(bear,n):5.1f}%")

    print(f"\n  IB direction -> premarket direction (column %, diagnostic, not tradeable)")
    for val in ['bullish', 'bearish']:
        s = sub[sub['ib_direction'] == val]
        n = len(s)
        bull = (s['premarket_direction'] == 'bullish').sum()
        bear = (s['premarket_direction'] == 'bearish').sum()
        print(f"    IB={val:<8} (n={n:4d})  -> premarket bullish {safe_pct(bull,n):5.1f}%  "
              f"premarket bearish {safe_pct(bear,n):5.1f}%")

    # premarket direction / close-half / size as outcomes vs post-IB
    print_subheader("Premarket direction / close-half -> Post-IB outcomes")
    for val in ['bullish', 'bearish']:
        subset = rdf[rdf['premarket_direction'] == val]
        print_outcome(f"premarket_direction={val}", subset)
        print_break(f"premarket_direction={val}", subset)
        collect_edge(f"premarket_direction={val}", subset, BASE_BULL_PCT)
    for val in ['upper', 'lower']:
        subset = rdf[rdf['premarket_close_half'] == val]
        print_outcome(f"premarket_close_half={val}", subset)
        collect_edge(f"premarket_close_half={val}", subset, BASE_BULL_PCT)

    print_header("PREMARKET -- DOES PREMARKET DIRECTION/SIZE SHARPEN THE ORB-30 SIGNAL?")
    print("  Baseline: ORB-30 close-half x its own internal extreme-first")
    print("  -> directional read on IB direction, live-safe at 10:00am.\n")

    print_subheader("+ premarket direction")
    for pm_dir in ['bullish', 'bearish']:
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['premarket_direction'] == pm_dir) & (rdf['orb30_close_half'] == ch) &
                             (rdf['orb30_internal_extreme_first'] == ext)]
                label = f"premarket_dir={pm_dir}+ch={ch}+ext={ext}"
                print_outcome(label, subset, direction_col='ib_direction')
                collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

    print_subheader("+ premarket size band")
    for band in ['Small', 'Average', 'Large']:
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = rdf[(rdf['premarket_range_pct_band'] == band) & (rdf['orb30_close_half'] == ch) &
                             (rdf['orb30_internal_extreme_first'] == ext)]
                label = f"premarket_size={band}+ch={ch}+ext={ext}"
                print_outcome(label, subset, direction_col='ib_direction')
                collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

    print_subheader("Baseline for comparison (no premarket factor)")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
            print_outcome(f"orb30={ch}+ext={ext}  (no premarket)", subset, direction_col='ib_direction')

    # ========================================================
    # RANKED EDGE TABLE
    # ========================================================
    print_header("RANKED EDGE TABLE -- ALL PATTERNS, SORTED BY EDGE")
    print(f"  Baseline Bull%: {BASE_BULL_PCT:.1f}  |  min n={MIN_DISPLAY_SAMPLE}\n")

    edge_df = pd.DataFrame(edge_rows).drop_duplicates(subset=['label']).sort_values(
        'edge_vs_base', ascending=False)
    print(f"  {'PATTERN':<58}{'n':>6}{'BULL%':>8}{'BEAR%':>8}{'EDGE':>8}{'BIAS':>6}")
    for _, row in edge_df.iterrows():
        print(f"  {row['label']:<58}{row['n']:>6}{row['bull_pct']:>7.1f}%{row['bear_pct']:>7.1f}%"
              f"{row['edge_vs_base']:>7.1f}%{row['bias']:>6}")

    edge_df.to_csv(RANKED_EDGE_PATH, index=False)
    print(f"\nSaved -> {RANKED_EDGE_PATH}")

    print_header("ANALYSIS COMPLETE")


# ============================================================
# ORB RETEST ANALYSIS -- levels revisited after the ORB closes,
# before IB completes (10:30am)
# ============================================================
# Question: given a live-safe directional bias at the ORB close
# (direction2 = low-first+upper-close -> bullish, high-first+lower-close ->
# bearish), does waiting for a pullback INTO the ORB's own range before
# entering make the signal MORE reliable, less reliable, or just delay an
# entry that was going to work anyway? And what do you give up on days
# where no pullback ever comes?
#
# Levels tested (all already in the Stage 2 dataset, no raw-bar reconstruction
# needed for the levels themselves -- only the touch-scan needs raw bars):
#   mid          -- midpoint of the ORB's own high-low range
#   high         -- the ORB's high (breakout-retest level in a bullish setup,
#                   since 'high formed first' is excluded by definition in
#                   bullish direction2 -- the high is whatever printed AFTER
#                   the low, so this level sits inside/at the edge of the ORB)
#   low          -- the ORB's low (deepest, highest-R:R pullback level;
#                   IS the origin extreme in a bullish setup)
#   vwap_session -- session VWAP sampled at that ORB's close
#
# 'Touch' = the level fell within some 1-min bar's [low, high] at any point
# between the ORB's close and IB completion. MAE% = worst move against the
# bias after that touch (before IB completes), as a % of the ORB's own
# range -- a rough stop-sizing reference, not a full R-multiple backtest.

def build_retest_dataset(bars, rdf):
    print_header("BUILDING ORB RETEST DATASET")
    print("  Scanning ORB-close -> IB-completion (10:30am) windows for touches")
    print("  of each ORB's own mid / high / low / session-VWAP levels...\n")

    rdf = rdf.copy()
    rdf['orb15_mid'] = (rdf['orb15_high'] + rdf['orb15_low']) / 2
    rdf['orb30_mid'] = (rdf['orb30_high'] + rdf['orb30_low']) / 2

    orb_specs = [
        ('ORB-15', 'orb15_direction2', 9, 45,
         {'mid': 'orb15_mid', 'high': 'orb15_high', 'low': 'orb15_low',
          'vwap_session': 'vwap_session_at_orb15'},
         'orb15_high', 'orb15_low'),
        ('ORB-30', 'orb30_direction2', 10, 0,
         {'mid': 'orb30_mid', 'high': 'orb30_high', 'low': 'orb30_low',
          'vwap_session': 'vwap_session_at_orb30'},
         'orb30_high', 'orb30_low'),
    ]

    records = []
    for i, (_, row) in enumerate(rdf.iterrows(), 1):
        if i % 500 == 0:
            print(f"  Day {i} / {len(rdf)} ...")

        actual = pd.Timestamp(row['date']) + pd.Timedelta(hours=7)
        ib_end = actual.replace(hour=10, minute=30)

        for orb_label, bias_col, end_h, end_m, level_cols, high_col, low_col in orb_specs:
            bias = row[bias_col]
            if bias not in ('bullish', 'bearish'):
                continue  # only score days with a clean direction2 signal

            orb_range = row[high_col] - row[low_col]
            if pd.isna(orb_range) or orb_range <= 0:
                continue

            orb_end = actual.replace(hour=end_h, minute=end_m)
            window = bars.loc[orb_end + pd.Timedelta(minutes=1): ib_end]

            rec = {'date': row['date'], 'orb_label': orb_label, 'bias': bias,
                   'ib_direction': row['ib_direction'],
                   'post_ib_direction': row['post_ib_direction']}

            if window.empty:
                for name in level_cols:
                    rec[f'{name}_touched'] = False
                    rec[f'{name}_minutes'] = np.nan
                    rec[f'{name}_mae_pct'] = np.nan
                records.append(rec)
                continue

            lows = window['low'].to_numpy()
            highs = window['high'].to_numpy()
            idx = window.index
            t0 = idx[0]

            for name, col in level_cols.items():
                level = row[col]
                if pd.isna(level):
                    rec[f'{name}_touched'] = False
                    rec[f'{name}_minutes'] = np.nan
                    rec[f'{name}_mae_pct'] = np.nan
                    continue
                mask = (lows <= level) & (highs >= level)
                if not mask.any():
                    rec[f'{name}_touched'] = False
                    rec[f'{name}_minutes'] = np.nan
                    rec[f'{name}_mae_pct'] = np.nan
                    continue
                first_pos = int(np.argmax(mask))
                rec[f'{name}_touched'] = True
                rec[f'{name}_minutes'] = (idx[first_pos] - t0).total_seconds() / 60.0

                if bias == 'bullish':
                    worst = lows[first_pos:].min()
                    mae_points = max(level - worst, 0.0)
                else:
                    worst = highs[first_pos:].max()
                    mae_points = max(worst - level, 0.0)
                rec[f'{name}_mae_pct'] = (mae_points / orb_range) * 100

            records.append(rec)

    retest_df = pd.DataFrame(records)
    print(f"\n  Built {len(retest_df):,} orb-day retest records: "
          f"{retest_df['orb_label'].value_counts().to_dict()}")
    retest_df.to_csv(RETEST_OUTPUT_PATH, index=False)
    print(f"  Saved -> {RETEST_OUTPUT_PATH}")
    return retest_df


def analyze_retests(retest_df):
    print_header("ORB-15 / ORB-30 RETEST ANALYSIS")
    print("  bias = direction2 (low-first+upper-close=bullish, high-first+lower-close=bearish)")
    print("  touch = level fell within a 1-min bar's range between ORB close and IB completion")
    print("  IB_match  = % of touched days where IB direction still confirms the bias")
    print("  edge      = IB_match(touched) minus the unconditional IB_match for that bias")
    print("              (positive = waiting for this retest IMPROVES the odds;")
    print("               negative = waiting for it selects AGAINST the better trades)")
    print("  median_MAE = median worst move against you after the touch, as a % of")
    print("              that ORB's own range (rough stop-sizing reference)\n")

    LEVELS = ['mid', 'high', 'low', 'vwap_session']
    summary_rows = []

    for orb_label in ['ORB-15', 'ORB-30']:
        odf = retest_df[retest_df['orb_label'] == orb_label]
        print_header(f"{orb_label} RETEST STATISTICS", width=95)

        for bias in ['bullish', 'bearish']:
            bdf = odf[odf['bias'] == bias]
            n_total = len(bdf)
            base_match = safe_pct((bdf['ib_direction'] == bias).sum(), n_total)
            print_subheader(f"{orb_label} bias={bias}  "
                            f"(n={n_total}, unconditional IB_match={base_match:.1f}%)")

            for level in LEVELS:
                touched = bdf[bdf[f'{level}_touched'] == True]
                untouched = bdf[bdf[f'{level}_touched'] == False]
                n_t, n_u = len(touched), len(untouched)

                if n_t < MIN_DISPLAY_SAMPLE:
                    print(f"    {level:<14} n_touched={n_t:4d}  (below min sample)")
                    continue

                touch_rate = safe_pct(n_t, n_total)
                ib_match_t = safe_pct((touched['ib_direction'] == bias).sum(), n_t)
                post_match_t = safe_pct((touched['post_ib_direction'] == bias).sum(), n_t)
                edge = ib_match_t - base_match
                med_min = touched[f'{level}_minutes'].median()
                med_mae = touched[f'{level}_mae_pct'].median()

                print(f"    {level:<14} touch_rate={touch_rate:5.1f}%  n={n_t:4d}  "
                      f"IB_match={ib_match_t:5.1f}%  edge={edge:+5.1f}%  "
                      f"post_IB_match={post_match_t:5.1f}%  "
                      f"median_wait={med_min:4.1f}m  median_MAE={med_mae:5.1f}%")

                if n_u >= MIN_DISPLAY_SAMPLE:
                    ib_match_u = safe_pct((untouched['ib_direction'] == bias).sum(), n_u)
                    print(f"    {'':<14} not touched: n={n_u:4d}  IB_match={ib_match_u:5.1f}%  "
                          f"(what you'd give up waiting for this level)")

                summary_rows.append({
                    'orb': orb_label, 'bias': bias, 'level': level,
                    'n_touched': n_t, 'touch_rate': touch_rate,
                    'ib_match': ib_match_t, 'edge_vs_unconditional': edge,
                    'post_ib_match': post_match_t,
                    'median_wait_min': med_min, 'median_mae_pct': med_mae,
                })
        print()

    print_header("RANKED SUMMARY -- ALL ORB x BIAS x LEVEL COMBINATIONS, BY EDGE")
    if not summary_rows:
        print("  No level/bias combination reached MIN_DISPLAY_SAMPLE -- nothing to rank.")
        print("  (This is expected on a small test dataset; on the full 15y file each")
        print("  combination should have hundreds to low-thousands of touched days.)")
        print_header("RETEST ANALYSIS COMPLETE")
        return

    summary_df = pd.DataFrame(summary_rows).sort_values('edge_vs_unconditional', ascending=False)
    print(f"  {'ORB':<8}{'BIAS':<10}{'LEVEL':<14}{'n':>6}{'TOUCH%':>8}{'IB_MATCH%':>11}"
          f"{'EDGE':>8}{'WAIT(m)':>9}{'MAE%':>7}")
    for _, r in summary_df.iterrows():
        print(f"  {r['orb']:<8}{r['bias']:<10}{r['level']:<14}{r['n_touched']:>6}"
              f"{r['touch_rate']:>7.1f}%{r['ib_match']:>10.1f}%{r['edge_vs_unconditional']:>+7.1f}%"
              f"{r['median_wait_min']:>8.1f}{r['median_mae_pct']:>6.1f}%")

    summary_df.to_csv(RETEST_EDGE_PATH, index=False)
    print(f"\nSaved -> {RETEST_EDGE_PATH}")
    print_header("RETEST ANALYSIS COMPLETE")


# ============================================================
# ORB-30 BREAKOUT-RETEST BACKTEST
# ============================================================
# Entry: first retest of ORB-30's own far extreme (high for bullish setups,
# low for bearish -- the level the retest analysis showed has the strongest,
# tightest-risk edge) in the window (ORB-30 close, IB completion].
# Fill assumption: filled exactly at the level (resting limit order, no
# slippage modeled).
# Stop: tested across BACKTEST_STOP_FRACTIONS, as a fraction of that day's
# ORB-30 range, placed beyond the entry level.
# Target: tested across BACKTEST_TARGET_R_MULTIPLES, as R-multiples of the
# stop distance.
# Exit windows: 'ib_completion' (hold to 10:30, mark-to-market at the IB's
# actual close if neither stop nor target is hit) and 'full_session' (hold
# to 4:00pm, mark-to-market at the session close).
# Conservative assumption: if a single 1-min bar's range would hit both stop
# and target, the STOP is scored as hit first -- there's no intrabar
# sequencing available at 1-min resolution, and assuming the worse outcome
# avoids overstating the edge.

def build_orb30_backtest(bars, rdf):
    print_header("BUILDING ORB-30 BREAKOUT-RETEST BACKTEST")
    print(f"  Stop fractions (of ORB-30 range): {BACKTEST_STOP_FRACTIONS}")
    print(f"  Target R-multiples: {BACKTEST_TARGET_R_MULTIPLES}")
    print("  Windows: ib_completion (to 10:30) and full_session (to 4:00pm)\n")

    results = defaultdict(list)  # (stop_frac, target_r, window) -> list of R outcomes
    n_qualifying = 0
    n_triggered = 0

    for i, (_, row) in enumerate(rdf.iterrows(), 1):
        if i % 500 == 0:
            print(f"  Day {i} / {len(rdf)} ...")

        bias = row['orb30_direction2']
        if bias not in ('bullish', 'bearish'):
            continue
        n_qualifying += 1

        sign = 1 if bias == 'bullish' else -1
        entry_level = row['orb30_high'] if bias == 'bullish' else row['orb30_low']
        orb_range = row['orb30_high'] - row['orb30_low']
        if pd.isna(entry_level) or pd.isna(orb_range) or orb_range <= 0:
            continue

        actual = pd.Timestamp(row['date']) + pd.Timedelta(hours=7)
        orb30_end = actual.replace(hour=10, minute=0)
        ib_end = actual.replace(hour=10, minute=30)
        session_end = actual.replace(hour=16, minute=0)

        trigger_window = bars.loc[orb30_end + pd.Timedelta(minutes=1): ib_end]
        if trigger_window.empty:
            continue
        t_lows = trigger_window['low'].to_numpy()
        t_highs = trigger_window['high'].to_numpy()
        touch_mask = (t_lows <= entry_level) & (t_highs >= entry_level)
        if not touch_mask.any():
            continue  # no retest -> no trade this day

        entry_time = trigger_window.index[int(np.argmax(touch_mask))]
        n_triggered += 1

        fwd_start = entry_time + pd.Timedelta(minutes=1)
        ib_fwd = bars.loc[fwd_start: ib_end]
        full_fwd = bars.loc[fwd_start: session_end]

        ib_close_price = row['ib_close']
        session_close_price = full_fwd['close'].iloc[-1] if not full_fwd.empty else ib_close_price

        for stop_frac in BACKTEST_STOP_FRACTIONS:
            stop_distance = stop_frac * orb_range
            for target_r in BACKTEST_TARGET_R_MULTIPLES:
                target_price = entry_level + sign * target_r * stop_distance
                stop_price = entry_level - sign * stop_distance

                for window_name, fwd, mtm_price in [
                    ('ib_completion', ib_fwd, ib_close_price),
                    ('full_session', full_fwd, session_close_price),
                ]:
                    f_lows = fwd['low'].to_numpy()
                    f_highs = fwd['high'].to_numpy()

                    if sign == 1:
                        stop_mask = f_lows <= stop_price
                        target_mask = f_highs >= target_price
                    else:
                        stop_mask = f_highs >= stop_price
                        target_mask = f_lows <= target_price

                    stop_pos = int(np.argmax(stop_mask)) if stop_mask.any() else None
                    target_pos = int(np.argmax(target_mask)) if target_mask.any() else None

                    if stop_pos is not None and (target_pos is None or stop_pos <= target_pos):
                        r_outcome = -1.0
                    elif target_pos is not None:
                        r_outcome = target_r
                    else:
                        r_outcome = sign * (mtm_price - entry_level) / stop_distance

                    results[(stop_frac, target_r, window_name)].append(r_outcome)

    print(f"\n  Qualifying ORB-30 direction2 days: {n_qualifying:,}")
    print(f"  Days the breakout-retest actually triggered: {n_triggered:,} "
          f"({safe_pct(n_triggered, n_qualifying):.1f}%)")
    return results, n_qualifying, n_triggered


def analyze_backtest(results, n_qualifying, n_triggered):
    print_header("ORB-30 BREAKOUT-RETEST BACKTEST -- RESULTS")
    print(f"  Qualifying signal days: {n_qualifying:,}  |  Triggered trades: {n_triggered:,} "
          f"({safe_pct(n_triggered, n_qualifying):.1f}%)")
    print("  Fill assumption: entry filled exactly at the level, no slippage.")
    print("  Tie assumption: same-bar stop+target -> scored as stop hit first.\n")

    rows = []
    for (stop_frac, target_r, window), r_list in results.items():
        arr = np.array(r_list)
        n = len(arr)
        win_rate = safe_pct((arr == target_r).sum(), n)
        stop_rate = safe_pct((arr == -1.0).sum(), n)
        timeout_rate = 100 - win_rate - stop_rate
        rows.append({
            'stop_pct_of_orb30_range': stop_frac * 100, 'target_R': target_r, 'window': window,
            'n': n, 'win_rate': win_rate, 'stop_rate': stop_rate, 'timeout_rate': timeout_rate,
            'avg_R': arr.mean(), 'total_R': arr.sum(), 'median_R': np.median(arr),
        })

    summary = pd.DataFrame(rows).sort_values('avg_R', ascending=False)
    print(f"  {'STOP%':>7}{'TARGET':>9}{'WINDOW':>16}{'n':>6}{'WIN%':>8}{'STOP%':>8}"
          f"{'T/OUT%':>8}{'AVG_R':>9}{'MED_R':>8}{'TOTAL_R':>10}")
    for _, r in summary.iterrows():
        print(f"  {r['stop_pct_of_orb30_range']:>6.0f}%{r['target_R']:>8.1f}R"
              f"{r['window']:>16}{r['n']:>6}{r['win_rate']:>7.1f}%{r['stop_rate']:>7.1f}%"
              f"{r['timeout_rate']:>7.1f}%{r['avg_R']:>+8.2f}R{r['median_R']:>+7.2f}R"
              f"{r['total_R']:>+9.1f}R")

    summary.to_csv(BACKTEST_RESULTS_PATH, index=False)
    print(f"\nSaved -> {BACKTEST_RESULTS_PATH}")
    print_header("BACKTEST COMPLETE")
    return summary


# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    need_bars = REBUILD_DATASET or RUN_RETEST_ANALYSIS or RUN_ORB30_BACKTEST \
        or not os.path.exists(OUTPUT_DATASET_PATH)
    bars = has_volume = None
    if need_bars:
        bars, has_volume = load_bars()

    if REBUILD_DATASET or not os.path.exists(OUTPUT_DATASET_PATH):
        rdf = build_dataset(bars, has_volume)
    else:
        print(f"Loading cached dataset from {OUTPUT_DATASET_PATH} ...")
        rdf = pd.read_csv(OUTPUT_DATASET_PATH, parse_dates=['date'])
        # infer whether VWAP columns were populated when the cache was built
        has_volume = rdf['ib_vs_vwap_weekly'].notna().any() \
            if 'ib_vs_vwap_weekly' in rdf.columns else False
        print(f"  {len(rdf):,} days loaded")

    run_analysis(rdf, has_volume)

    if RUN_RETEST_ANALYSIS:
        retest_df = build_retest_dataset(bars, rdf)
        analyze_retests(retest_df)

    if RUN_ORB30_BACKTEST:
        backtest_results, n_qual, n_trig = build_orb30_backtest(bars, rdf)
        analyze_backtest(backtest_results, n_qual, n_trig)


                            STAGE 1 -- LOADING 15y 1-MIN OHLCV DATA                            

  Loading data from: C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
  Loaded 4,161,727 bars | 2013-01-02 06:00:00 -> 2025-03-14 16:59:00
  Volume column present: True

                      STAGE 2 -- BUILDING PER-DAY PRICE-STRUCTURE DATASET                      
  VWAP preprocessing done: 637 weeks, 3783 sessions
  3786 candidate trading days
  Day 250 / 3785 ...
  Day 500 / 3785 ...
  Day 750 / 3785 ...
  Day 1000 / 3785 ...
  Day 1250 / 3785 ...
  Day 1500 / 3785 ...
  Day 1750 / 3785 ...
  Day 2000 / 3785 ...
  Day 2250 / 3785 ...
  Day 2500 / 3785 ...
  Day 2750 / 3785 ...
  Day 3000 / 3785 ...
  Day 3250 / 3785 ...
  Day 3500 / 3785 ...
  Day 3750 / 3785 ...

Built 3,109 daily records | skipped 676

Size band distribution (IB range %):
Average       1353
Small         1074
Large          448
Very Large     232
Very Small       2
Name: ib_range_pct_band, dtype: int64

Structural sc

    ORB-30=upper+orb30_internal_extreme_first=low      n=1379  Bull: 83.8%  Bear: 16.2%  Edge:33.8%  [BULL]
    ORB-30=lower+orb30_internal_extreme_first=high     n=1180  Bull: 16.9%  Bear: 83.1%  Edge:33.1%  [BEAR]
    ORB-30=lower+orb30_internal_extreme_first=low      n= 206  Bull: 48.5%  Bear: 51.5%  Edge: 1.5%  [BEAR]

-----------------------------------------------------------------------------------------------
  ORB-15 close-half x ORB-15's OWN extreme-first (LIVE-SAFE at 9:45am) -> IB direction
-----------------------------------------------------------------------------------------------
    ORB-15=upper+orb15_internal_extreme_first=high     n= 316  Bull: 48.1%  Bear: 51.9%  Edge: 1.9%  [BEAR]
    ORB-15=upper+orb15_internal_extreme_first=low      n=1353  Bull: 74.8%  Bear: 25.2%  Edge:24.8%  [BULL]
    ORB-15=lower+orb15_internal_extreme_first=high     n=1173  Bull: 26.3%  Bear: 73.7%  Edge:23.7%  [BEAR]
    ORB-15=lower+orb15_internal_extreme_first=low      n= 262  Bull: 55.

        ORB-30 below                                   n=1235  Bull: 53.8%  Bear: 46.2%  Edge: 3.8%  [BULL]
        ORB-30 below                                   n=1239  H1h= 35.0%  L1h= 39.5%  Hses= 56.7%  Lses= 60.5%

  IB vs weekly VWAP  (n valid = 3109)
        IB above                                       n=1855  Bull: 56.8%  Bear: 43.2%  Edge: 6.8%  [BULL]
        IB above                                       n=1860  H1h= 51.0%  L1h= 25.9%  Hses= 71.1%  Lses= 46.9%
        IB below                                       n=1246  Bull: 53.7%  Bear: 46.3%  Edge: 3.7%  [BULL]
        IB below                                       n=1249  H1h= 26.9%  L1h= 48.3%  Hses= 50.3%  Lses= 66.1%

-----------------------------------------------------------------------------------------------
  Session (9:30 anchor, running -- classic day-VWAP)
-----------------------------------------------------------------------------------------------

  ORB-15 vs session VWAP  (n valid = 3109)
        ORB


  premarket size = Average (n=1566)
        ib=upper+ext=high                              n= 196  H1h= 55.1%  L1h= 19.4%  Hses= 78.6%  Lses= 44.9%
        ib=upper+ext=low                               n= 664  H1h= 69.7%  L1h= 11.4%  Hses= 83.9%  Lses= 32.5%
        ib=lower+ext=high                              n= 586  H1h=  9.4%  L1h= 62.8%  Hses= 34.0%  Lses= 77.6%
        ib=lower+ext=low                               n= 120  H1h= 25.0%  L1h= 51.7%  Hses= 54.2%  Lses= 73.3%

  premarket size = Large (n=366)
        ib=upper+ext=high                              n=  47  H1h= 57.4%  L1h= 17.0%  Hses= 85.1%  Lses= 38.3%
        ib=upper+ext=low                               n= 141  H1h= 64.5%  L1h= 12.8%  Hses= 84.4%  Lses= 34.0%
        ib=lower+ext=high                              n= 144  H1h= 12.5%  L1h= 70.1%  Hses= 41.7%  Lses= 81.9%
        ib=lower+ext=low                               n=  34  H1h= 26.5%  L1h= 61.8%  Hses= 70.6%  Lses= 73.5%

       PREMARKET -- DOES PREMARK

  Day 500 / 3109 ...
  Day 1000 / 3109 ...
  Day 1500 / 3109 ...
  Day 2000 / 3109 ...
  Day 2500 / 3109 ...
  Day 3000 / 3109 ...

  Built 5,085 orb-day retest records: {'ORB-30': 2559, 'ORB-15': 2526}
  Saved -> C:\Users\tammm\NQ and ES analysis\nq_15y_orb_retest_dataset.csv

                                ORB-15 / ORB-30 RETEST ANALYSIS                                
  bias = direction2 (low-first+upper-close=bullish, high-first+lower-close=bearish)
  touch = level fell within a 1-min bar's range between ORB close and IB completion
  IB_match  = % of touched days where IB direction still confirms the bias
  edge      = IB_match(touched) minus the unconditional IB_match for that bias
              (positive = waiting for this retest IMPROVES the odds;
               negative = waiting for it selects AGAINST the better trades)
  median_MAE = median worst move against you after the touch, as a % of
              that ORB's own range (rough stop-sizing reference)


                   

  Day 500 / 3109 ...
  Day 1000 / 3109 ...
  Day 1500 / 3109 ...
  Day 2000 / 3109 ...
  Day 2500 / 3109 ...
  Day 3000 / 3109 ...

  Qualifying ORB-30 direction2 days: 2,559
  Days the breakout-retest actually triggered: 1,817 (71.0%)

                           ORB-30 BREAKOUT-RETEST BACKTEST -- RESULTS                          
  Qualifying signal days: 2,559  |  Triggered trades: 1,817 (71.0%)
  Fill assumption: entry filled exactly at the level, no slippage.
  Tie assumption: same-bar stop+target -> scored as stop hit first.

    STOP%   TARGET          WINDOW     n    WIN%   STOP%  T/OUT%    AVG_R   MED_R   TOTAL_R
      35%     3.0R    full_session  1817   23.9%   69.0%    7.0%   +0.11R  -1.00R   +202.4R
      50%     2.0R    full_session  1817   29.8%   60.0%   10.1%   +0.06R  -1.00R   +102.7R
      50%     3.0R    full_session  1817   16.7%   64.1%   19.2%   +0.06R  -1.00R   +100.5R
      15%     1.5R   ib_completion  1817   38.7%   54.3%    7.0%   +0.05R  -1.00R    +94.9R
   

In [2]:
"""
Does where the IB closes inside its own range predict which side breaks first?

Standalone - no imports beyond pandas/numpy/scipy. Set FILE_PATH below.

IB      = 09:30-10:30 ET
Post    = 10:30-16:00 ET, same session
Close   = (ib_close - ib_low) / (ib_high - ib_low), so 0.0 = closed on the low,
          1.0 = closed on the high
Break   = the first post-IB 1-minute bar whose high exceeds ib_high, versus the
          first whose low falls below ib_low, resolved at 1-minute resolution.

The headline table looks like a powerful signal. The null-model section at the
bottom shows it is not - see the note there before using any of it.
"""
import os

import numpy as np
import pandas as pd

# ---- configure -------------------------------------------------------------
FILE_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
CACHE = "nq_1min_cache.pkl"     # built on first run, reused after
DAY_SHIFT_HOURS = 6             # 18:00 ET rolls onto the next session date
# ----------------------------------------------------------------------------

IB_START, IB_END, RTH_END = 570, 630, 960          # 09:30, 10:30, 16:00 in mins
MIN_IB_BARS, MIN_POST_BARS = 50, 250               # drops half-days and gaps


def load_minute_data(file_path, day_shift_hours=DAY_SHIFT_HOURS):
    """Same loader as the main study, so results are directly comparable."""
    head = pd.read_csv(file_path, nrows=0)
    low = {c.lower(): c for c in head.columns}
    wanted = [low[c] for c in
              ("date", "time", "datetime", "timestamp", "open", "high", "low", "close")
              if c in low]
    df = pd.read_csv(
        file_path, usecols=wanted,
        dtype={low[c]: "float64" for c in ("open", "high", "low", "close") if c in low},
    )
    if "date" in low and "time" in low:
        dates = pd.to_datetime(df.pop(low["date"]), format="%Y-%m-%d", cache=True)
        dt = dates + pd.to_timedelta(df.pop(low["time"]))
    else:
        key = low.get("datetime") or low["timestamp"]
        dt = pd.to_datetime(df.pop(key), cache=True)
    df = df.rename(columns={low[s]: s.capitalize()
                            for s in ("open", "high", "low", "close")})
    df = df.assign(Datetime=dt).set_index("Datetime").sort_index()
    df = df[["Open", "High", "Low", "Close"]]
    df = df[~df.index.duplicated(keep="first")].dropna()
    df["session_date"] = (df.index + pd.Timedelta(hours=day_shift_hours)).normalize()
    df["tod"] = df.index.hour * 60 + df.index.minute
    df["dow"] = df["session_date"].dt.dayofweek
    return df


def get_data(file_path=FILE_PATH, cache=CACHE):
    if cache and os.path.exists(cache):
        return pd.read_pickle(cache)
    df = load_minute_data(file_path)
    if cache:
        df.to_pickle(cache)
    return df


def build(df):
    rth = df[(df["tod"] >= IB_START) & (df["tod"] < RTH_END) & (df["dow"] < 5)]
    ib, post = rth[rth["tod"] < IB_END], rth[rth["tod"] >= IB_END]

    g = ib.groupby("session_date")
    out = pd.DataFrame({
        "ib_high": g["High"].max(), "ib_low": g["Low"].min(),
        "ib_open": g["Open"].first(), "ib_close": g["Close"].last(),
        "ib_bars": g["Close"].size(),
    })

    rows = []
    for sd, p in post.groupby("session_date"):
        if sd not in out.index:
            continue
        r = out.loc[sd]
        if r.ib_bars < MIN_IB_BARS or len(p) < MIN_POST_BARS:
            continue
        rng = r.ib_high - r.ib_low
        if rng <= 0:
            continue

        hi_hit = np.flatnonzero(p["High"].to_numpy() > r.ib_high)
        lo_hit = np.flatnonzero(p["Low"].to_numpy() < r.ib_low)
        i_hi = hi_hit[0] if len(hi_hit) else None
        i_lo = lo_hit[0] if len(lo_hit) else None

        if i_hi is None and i_lo is None:
            first = "neither"
        elif i_lo is None or (i_hi is not None and i_hi < i_lo):
            first = "high"
        elif i_hi is None or i_lo < i_hi:
            first = "low"
        else:
            first = "same_minute"

        rows.append({"session_date": sd,
                     "close_pos": (r.ib_close - r.ib_low) / rng,
                     "ib_range_pct": rng / r.ib_open * 100,
                     "first": first})
    return pd.DataFrame(rows).set_index("session_date")


def decile_table(d):
    edges = np.arange(0, 1.01, 0.1)
    d = d.copy()
    d["b"] = np.clip(np.digitize(d["close_pos"], edges[1:-1]), 0, 9)
    rows = []
    for b, s in d.groupby("b"):
        c = s["first"].value_counts()
        hi, lo = c.get("high", 0), c.get("low", 0)
        dec = hi + lo
        p = hi / dec * 100 if dec else np.nan
        se = np.sqrt(p / 100 * (1 - p / 100) / dec) * 100 if dec else np.nan
        rows.append({"bucket": f"{edges[b]:.1f}-{edges[b+1]:.1f}", "n": len(s),
                     "high": hi, "low": lo, "neither": c.get("neither", 0),
                     "high_first_pct": p, "ci_lo": p - 1.96 * se,
                     "ci_hi": p + 1.96 * se})
    return pd.DataFrame(rows)


# ==================================================================
# Null-model comparison - read this before using the table above
# ==================================================================
# For a driftless random walk starting inside a range, the probability of
# touching the upper barrier before the lower one is the gambler's-ruin
# result: distance-to-lower / total distance. Since close_pos IS that
# fraction, the null prediction for P(high breaks first) is close_pos
# itself - the same number, containing no market behaviour at all.
#
# On 3,519 decided sessions the observed rates sit on top of that null:
# chi-square 9.11 on 10 df, p = 0.52, pooled difference +0.4 points. The
# apparent edge is pure geometry.
def null_comparison(d):
    dec = d[d["first"].isin(["high", "low"])].copy()
    dec["hi"] = (dec["first"] == "high").astype(int)
    edges = np.arange(0, 1.01, 0.1)
    dec["b"] = np.clip(np.digitize(dec["close_pos"], edges[1:-1]), 0, 9)
    rows, chi = [], 0.0
    for b, s in dec.groupby("b"):
        n, mp, obs = len(s), s["close_pos"].mean(), s["hi"].mean()
        se = np.sqrt(mp * (1 - mp) / n)
        z = (obs - mp) / se
        chi += z ** 2
        rows.append({"bucket": f"{edges[b]:.1f}-{edges[b+1]:.1f}", "n": n,
                     "null_pct": mp * 100, "obs_pct": obs * 100,
                     "diff": (obs - mp) * 100, "z": z})
    return pd.DataFrame(rows), chi, len(rows)


def run(file_path=FILE_PATH):
    df = get_data(file_path)
    d = build(df)
    print(f"sessions analysed: {len(d):,}   "
          f"{d.index.min():%Y-%m-%d} to {d.index.max():%Y-%m-%d}")
    print("outcomes:", d["first"].value_counts().to_dict(), "\n")

    print("IB close position within its own range -> which side breaks first")
    print("(0.0 = closed on its low, 1.0 = closed on its high)\n")
    print(decile_table(d).to_string(index=False,
          float_format=lambda x: f"{x:.1f}"))

    nc, chi, k = null_comparison(d)
    print("\n\nvs the gambler's-ruin null (null % = the bucket's own mean position)\n")
    print(nc.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    try:
        from scipy.stats import chi2
        p = 1 - chi2.cdf(chi, k)
        print(f"\nchi-square {chi:.2f} on {k} df, p = {p:.2f}")
    except ImportError:
        print(f"\nchi-square {chi:.2f} on {k} df")
    print("A non-significant result means close position predicts the break "
          "no better\nthan its distance to each barrier already implies.")
    return d


if __name__ == "__main__":
    run()

sessions analysed: 3,670   2010-06-07 to 2025-03-14
outcomes: {'high': 1912, 'low': 1607, 'neither': 151} 

IB close position within its own range -> which side breaks first
(0.0 = closed on its low, 1.0 = closed on its high)

 bucket   n  high  low  neither  high_first_pct  ci_lo  ci_hi
0.0-0.1 340    19  315        6             5.7    3.2    8.2
0.1-0.2 365    46  303       16            13.2    9.6   16.7
0.2-0.3 339    70  236       33            22.9   18.2   27.6
0.3-0.4 291    99  176       16            36.0   30.3   41.7
0.4-0.5 324   141  164       19            46.2   40.6   51.8
0.5-0.6 318   154  149       15            50.8   45.2   56.5
0.6-0.7 301   198   93       10            68.0   62.7   73.4
0.7-0.8 397   301   82       14            78.6   74.5   82.7
0.8-0.9 483   408   64       11            86.4   83.4   89.5
0.9-1.0 512   476   25       11            95.0   93.1   96.9


vs the gambler's-ruin null (null % = the bucket's own mean position)

 bucket   n  null_p